<font size="6">**Pre-allenare dati non etichettati**</font><br>
> *Copyright Antonio Piemontese 2026* (inspired to Sebastian Raschka's work).

Questo capitolo, passo dopo passo:

* in apertura, **ereditiamo il blocco di codice** dei notebook 2-4 (`GPTModel`, attenzione, data loader, `generate_text_simple`): è ciò che rende questo capitolo **autosufficiente**

* **costruiamo la valutazione numerica** del testo generato — la *loss* che guiderà il training:
    * partiamo dal perché l'occhio non basta: generiamo con il modello *non ancora allenato* e otteniamo testo privo di senso
    * ci dotiamo di due attrezzi di servizio, `text_to_token_ids` e `token_ids_to_text`, per passare agilmente dal testo ai tensori e ritorno — li riuseremo per tutto il capitolo
    * assembliamo passo-passo la **cross-entropy loss**: logits → softmax → probabilità dei token target → logaritmo → media → negazione (vedi slide notebook 4)
    ![](la-cross-entropy-loss.png)<br>
    * la rileggiamo come **perplexity**, la versione "interpretabile" della stessa quantità
* **prepariamo i dati** di training e di validazione:
    * carichiamo il racconto *"The Verdict"* (~20.000 token) e lo suddividiamo 90/10 tra training e validation set
    * riusiamo i data loader a finestra scorrevole (capitolo 2) e scriviamo le funzioni `calc_loss_batch` / `calc_loss_loader`
    * portiamo il calcolo sul **device** (CPU o GPU) e misuriamo le loss iniziali del modello non allenato
* **alleniamo l'LLM**, implementando la funzione di training:
    * scriviamo il ciclo `train_model_simple` con l'ottimizzatore **AdamW**, la valutazione periodica (`evaluate_model`) e un campione di testo a ogni epoca (`generate_and_print_sample`)
    * tracciamo il grafico delle loss su 10 epoche e diagnostichiamo l'**overfitting** (la memorizzazione del piccolo dataset)
* **controlliamo la casualità** del testo generato con le strategie di decoding:
    * tocchiamo con mano i limiti della scelta greedy (argmax)
    * introduciamo il **temperature scaling** e il campionamento con `torch.multinomial`
    * aggiungiamo il **top-k sampling** e scriviamo la nuova funzione `generate` che combina le due tecniche
* **salviamo e ricarichiamo i pesi** del modello:
    * serializziamo lo `state_dict` nei file `.pth`, con `torch.save` / `load_state_dict`
    * salviamo anche lo **stato dell'ottimizzatore**, per poter riprendere il training
* **carichiamo i pesi pre-addestrati di OpenAI** (GPT-2 124M), per **due vie**:
    * la via *alternativa*, che apre la sezione: scarichiamo i pesi **già convertiti in formato PyTorch** (`.pth`) dal repository di Raschka, senza alcuna dipendenza da TensorFlow
    * la via *storica*: scarichiamo i **checkpoint TensorFlow originali** tramite il modulo `gpt_download.py` (che preleviamo dal repository e verifichiamo con `ast.parse` prima dell'import) e mappiamo i pesi a mano con `load_weights_into_gpt`, spacchettamento della matrice Q/K/V e *weight tying* inclusi
    * **dimostriamo** infine che le due vie producono lo stesso identico modello: confrontiamo gli `state_dict`, parametro per parametro
* **generiamo testo coerente** con il modello pre-addestrato — e chiudiamo il cerchio aperto in testa al capitolo

Completano il capitolo le **note operative** 📌 (tra cui la diagnosi e la cura del conflitto OpenMP negli ambienti conda su Windows) e gli **innesti 🧩 gpt-oss**, che proiettano ogni tema — loss, scala dei dati e del calcolo, decoding, formato e fonte dei pesi — sul modello open-weight di OpenAI (il nostro ultimo passo del corso).

# Ordine didattico

Per velocizzare saltiamo per il momento il notebook 4 sull'architettura, tranne due celle importanti:
- la classe `TransformerBlock` ("l'attenzione, più una rete feedforward e due scorciatoie") 
- e la classe `GPTModel` ("questo blocco, impilato 12 volte, più embedding in ingresso e proiezione in uscita: fine, il modello è tutto qui"). 

Niente LayerNorm, GELU, gradiente evanescente: solo l'assemblaggio. Con quel ponte, quando nel 5 compare `GPTModel(GPT_CONFIG_124M)`, per la platea non è una scatola nera ma l'attenzione impacchettata.

---
**Legenda delle icone (standard) usate nel notebook**:<br>

👉 punto di attenzione, il "succo"<br>
📌 nota / inizio di una nota<br>
📦 punto elenco importante<br>
📊 dati/numeri<br>
🔹 punto elenco normale<br>
⭐ punto elenco importante<br>
✅ punto risolto, positivo<br>
❌ punto negativo, da evitare<br>
⚠️ attenzione, warning, allarme<br>
💡 idea chiave<br>
🧠 idea intuitiva<br>
📝 sintesi / bottom-line<br>
⟶ conseguenza, effetto, passo successivo

La seguente cella deve essere eseguita **per prima** nel notebook.

Autorizza il processo a convivere con due copie del runtime `OpenMP` <sup>1</sup> (quella di *torch-pip* e quella della *MKL* di conda <sup>2</sup>), che su Windows altrimenti ucciderebbero il kernel al primo grafico *matplotlib*. Va eseguita **prima di ogni import** — ed è una cintura di sicurezza.

<sup>1</sup> `OpenMP` (Open Multi-Processing): è lo standard con cui i programmi in C/C++/Fortran sfruttano più core della CPU in parallelo dentro uno stesso processo, usando i thread (flussi di esecuzione paralleli che condividono la memoria). In pratica: quando numpy moltiplica due matrici grandi e vedi tutti i core al lavoro, è OpenMP che sta distribuendo il calcolo.

<sup>2</sup> *MKL* sta per **Math Kernel Library**: è la libreria di routine matematiche ottimizzate di Intel — moltiplicazioni di matrici, FFT, algebra lineare (le funzioni BLAS/LAPACK) — scritta per spremere al massimo i processori. Non la si chiama mai direttamente: è il motore sotto il cofano di *numpy*; quando si fa `A @ B`, *numpy* delega il calcolo vero a lei.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>Gli "innesti gpt-oss"</b> <i>(convenzione introdotta nel notebook 2)</i><br><br>
Nell'agosto 2025 OpenAI ha rilasciato <b>gpt-oss-120b</b> e <b>gpt-oss-20b</b>, i suoi primi modelli <i>open-weight</i> (cioè "a pesi aperti": i pesi sono liberamente scaricabili e utilizzabili, con licenza Apache 2.0) dopo GPT-2.<br><br>
Le celle come questa — su <u>sfondo azzurro</u> e contrassegnate dal simbolo 🧩 — sono <b>innesti gpt-oss</b>: nei punti di contatto con il programma del corso, anticipano <u>come cambia in gpt-oss</u> il concetto appena studiato su GPT-2. Sono <b>letture a margine</b>: il filo del corso resta GPT-2, e il confronto completo sarà sviluppato nel <b>modulo dedicato a gpt-oss</b> (notebook 8).<br><br>
In questo capitolo gli innesti parlano soprattutto di <b>scala</b>: quanti dati, quante ore di calcolo, e che cosa resta davvero allenabile su hardware comune.
</div>

Ora è il momento di implementare una **funzione di training** e **pre-addestrare l'LLM**. Impareremo le <u>tecniche di base</u> per la valutazione del modello per misurare **la qualità del testo generato**, un requisito fondamentale per **l'ottimizzazione** dell'LLM durante il processo di training.

Inoltre, discuteremo come **caricare i pesi pre-addestrati**, fornendo al nostro LLM un solido punto di partenza per la messa a punto successiva [(*fine-tuning*)](https://it.wikipedia.org/wiki/Fine-tuning_(deep_learning)) dei notebook 6 e 7. La figura seguente illustra **il piano generale del corso**, già visto negli altri notebook, evidenziando gli argomenti di cui parleremo <u>in questo notebook</u>.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/chapter-overview.webp" width=800px>

La figura illustra le **tre fasi (*stage*) principali della codifica di un LLM**. Questo notebook si concentra sulla fase 2: il pre-allenamento dell'LLM (passo 4), che include l'implementazione del codice di allenamento (passo 5), la valutazione delle prestazioni (passo 6) e il salvataggio e il caricamento dei pesi del modello (passo 7).

**Nota sui parametri dei pesi (*weight parameters*)**<br>
Nel contesto degli LLM e di altri modelli di deep learning, i **pesi** si riferiscono ai **parametri allenabili** che il processo di allenamento "aggiusta" (ottimizza). Questi pesi sono anche noti come parametri dei pesi o semplicemente **parametri**. In framework come PyTorch, questi pesi sono memorizzati in layer lineari (*linear layer*); li abbiamo utilizzati per implementare il modulo di attenzione multi-head nel notebook 3 e il `GPTModel` nel notebook 4. Dopo aver inizializzato un livello (`new_layer = torch.nn.Linear(...))`, possiamo **accedere** ai suoi pesi tramite l'attributo `.weight`, ad esempio `new_layer.weight`. Inoltre, per comodità, PyTorch consente l'**accesso diretto** a tutti i parametri allenabili di un modello, inclusi pesi e bias, tramite il metodo **`model.parameters()`**, che utilizzeremo in seguito durante l'implementazione dell'allenamento del modello.

# Import dei moduli (classi e funzioni) degli altri notebook

La cella successiva contiene il codice Python contenuto nel seguente file (del sito):<br>
**ch05/01_main-chapter-code/previous_chapters.py**<br>
che definisce classi e funzioni dei notebook 2-3-4 necessari a questo notebook.

Si noti la guardia:

```python
    if __name__ == "__main__":
```

Il meccanismo: ogni volta che Python esegue un file, gli assegna una variabile automatica di modulo chiamata `__name__`, e il valore dipende da come il file viene eseguito: 
- se lo lanciamo direttamente (*python mio_script.py*, o eseguendo la cella), `__name__` vale la stringa speciale `"__main__"`; 
- se invece il file viene importato da un altro (*import mio_script*), `__name__` vale il nome del modulo, cioè "mio_script". 

Quella riga, quindi, si legge così: "esegui il blocco qui sotto solo se questo file è il protagonista della sessione, non quando viene importato come libreria".<br>
Senza quella guardia, un file che in fondo contiene una chiamata di prova — un download, un training, una stampa — la farebbe partire anche quando qualcun altro lo importa solo per prendersi una funzione. La guardia separa i due ruoli di uno stesso file.

In [5]:
# Questo file raccoglie tutto il codice rilevante che abbiamo trattato finora nei capitoli 2-4.
# Questo file può essere eseguito come script autonomo.
#
# Mappa del file: dati (notebook 2) → attenzione (notebook 3) → architettura e
# generazione (notebook 4) → in coda, un collaudo rapido eseguito solo da script.

import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#####################################
# Notebook 2
#####################################


# ------------------------------------------------------------------
# RECAP — GPTDatasetV1 (notebook 2)
# Trasforma il testo grezzo in coppie (input, target) per il compito
# "predici il prossimo token": con una finestra scorrevole ritagliamo
# sequenze di max_length token; il target è la stessa sequenza spostata
# di UNA posizione in avanti (lo "shift di 1" che dà significato a ogni
# riga dell'output del modello). Lo stride regola la sovrapposizione
# tra finestre consecutive.
# ------------------------------------------------------------------
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


# ------------------------------------------------------------------
# RECAP — create_dataloader_v1 (notebook 2)
# Funzione di comodo: istanzia il tokenizzatore BPE di GPT-2, costruisce
# il GPTDatasetV1 e lo impacchetta in un DataLoader PyTorch, che servirà
# i batch (mescolati, se shuffle=True) al ciclo di training.
# ------------------------------------------------------------------
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


#####################################
# Notebook 3
#####################################
# ------------------------------------------------------------------
# RECAP — MultiHeadAttention (notebook 3)
# Il cuore del modello. Per ogni token calcoliamo query, chiavi e valori
# (tre proiezioni lineari); i punteggi di attenzione sono prodotti scalari
# query·chiave, scalati per la radice di head_dim; la MASCHERA CAUSALE
# nasconde il futuro (ogni posizione vede solo se stessa e il passato);
# la softmax trasforma i punteggi in pesi, con cui facciamo la media
# pesata dei valori. Il "multi-head" è ottenuto senza cicli: con view e
# transpose spacchiamo le proiezioni in num_heads teste da head_dim
# dimensioni, calcoliamo tutte le attenzioni in un colpo solo, poi
# ricomponiamo e mescoliamo le teste con out_proj.
# ------------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec


#####################################
# Notebook 4
#####################################
# ------------------------------------------------------------------
# RECAP — LayerNorm (notebook 4)
# Normalizza ogni token lungo la dimensione degli embedding: media 0 e
# varianza 1 (varianza "biased", senza correzione di Bessel), poi due
# parametri appresi — scale e shift — ridanno libertà al modello.
# Serve a stabilizzare l'allenamento delle reti profonde.
# ------------------------------------------------------------------
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


# ------------------------------------------------------------------
# RECAP — GELU (notebook 4)
# La non-linearità del feedforward: una curva liscia, qui
# nell'approssimazione con tanh usata dal GPT-2 originale.
# ------------------------------------------------------------------
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


# ------------------------------------------------------------------
# RECAP — FeedForward (notebook 4)
# La "pancia" di ogni blocco: espansione ×4 (768 → 3072), GELU,
# contrazione (3072 → 768). Applicata a ogni token indipendentemente;
# è dove il blocco elabora ciò che l'attenzione ha raccolto.
# ------------------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


# ------------------------------------------------------------------
# RECAP — TransformerBlock (notebook 4)
# La ricetta che si ripete: norm → attenzione → (+ residuo),
# norm → feedforward → (+ residuo). Normalizzazione PRIMA di ogni
# sotto-modulo (pre-norm) e due shortcut connections per blocco,
# che tengono vivo il gradiente nelle reti profonde.
# ------------------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


# ------------------------------------------------------------------
# RECAP — GPTModel (notebook 4)
# L'assemblaggio completo: embedding dei token + embedding delle
# posizioni → dropout → pila di n_layers TransformerBlock → norm
# finale → testa lineare verso il vocabolario. L'output sono i LOGITS
# di forma [batch, num_token, 50257]: una riga di punteggi per ogni
# posizione — la materia prima della loss (questo capitolo).
# ------------------------------------------------------------------
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


# ------------------------------------------------------------------
# RECAP — generate_text_simple (notebook 4)
# Il ciclo autoregressivo nella versione "greedy": ritagliamo il
# contesto agli ultimi context_size token, prendiamo i logits
# dell'ULTIMA posizione (l'unica che ha visto tutto), scegliamo il
# token più probabile con argmax, lo accodiamo e ripetiamo.
# Un passaggio completo nel modello per ogni token generato.
# ------------------------------------------------------------------
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


# ------------------------------------------------------------------
# RECAP — Collaudo da script (la "guardia")
# Il blocco sotto gira SOLO lanciando questo file direttamente
# (python previous_chapters.py), NON quando il notebook lo importa:
# è la doppia natura libreria/programma vista a lezione. Il collaudo
# crea un modello NON allenato e genera 10 token: il testo privo di
# senso che otterremo è atteso — ed è il punto di partenza del capitolo.
# ------------------------------------------------------------------
if __name__ == "__main__":

    # La configurazione del 124M, con il contesto RIDOTTO a 256
    # per rendere leggero il training di questo capitolo
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 256,   # Context length (modified wrt notebook 4 (1024))
        "emb_dim": 768,          # Embedding dimension
        "n_heads": 12,           # Number of attention heads
        "n_layers": 12,          # Number of layers
        "drop_rate": 0.1,        # Dropout rate
        "qkv_bias": False        # Query-Key-Value bias
    }

    # Seed per la riproducibilità; eval() spegne il dropout
    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_124M)
    model.eval()  # disable dropout

    # Codifichiamo il prompt in ID di token e aggiungiamo la dimensione batch
    start_context = "Hello, I am"

    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)

    print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
    print("\nInput text:", start_context)
    print("Encoded input text:", encoded)
    print("encoded_tensor.shape:", encoded_tensor.shape)

    # Generazione greedy di 10 token con il modello non allenato
    out = generate_text_simple(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=10,
        context_size=GPT_CONFIG_124M["context_length"]
    )
    decoded_text = tokenizer.decode(out.squeeze(0).tolist())

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", out)
    print("Output length:", len(out[0]))
    print("Output text:", decoded_text)


                      IN

Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


                      OUT

Output: tensor([[15496,    11,   314,   716, 13240, 11381,  4307,  7640, 16620, 34991,
          6842, 37891, 19970, 47477]])
Output length: 14
Output text: Hello, I am Laur inhab DistrinetalkQueue bear confidentlyggyenium


---
Come si è visto la `context_length` è stata ridotta da 1.024 a **256** — ed è una scelta di **economia del training**, presa apposta per questo notebook. Vediamo perché conviene, cosa cambia nei conti, e perché è legittima.

**Perché conviene.** Il `context_length` è la finestra massima che il modello elabora in un passaggio, e nel training domina i costi su due fronti. Primo, **l'attenzione è quadratica nella lunghezza**: ogni blocco calcola una matrice di punteggi `n × n` per testa — a 1.024 token sono 1.048.576 celle per testa per blocco; a 256 sono 65.536, **sedici volte meno** calcolo e memoria per le attivazioni, moltiplicato per 12 teste e 12 blocchi. Secondo, con la finestra scorrevole del `GPTDatasetV1` (che usa `max_length = context_length`), sequenze più corte significano **batch più leggeri**. Sommando: il capitolo deve far vedere un training che *finisce davanti agli occhi* — i famosi 11 minuti sulla vecchia GPU, oggi molto meno sulla 5090 — e 256 è il compromesso che lo rende possibile anche sul portatile di un corsista o su Colab gratuito.

**C'è anche una ragione di dati, più sottile**: "The Verdict" ha ~20.000 token. Con finestre da 1.024 e lo stride pari alla finestra <sup>1</sup>, il training set si ridurrebbe a una **manciata di esempi per epoca** (una ventina scarsa); con 256 ne ottieni quattro volte tanti, batch più numerosi, loss più stabile. Su un dataset microscopico, la finestra grande è perfino controproducente.

**Perché è legittimo** — ed un punto concettuale: il `context_length` **non cambia il numero di parametri "di conoscenza"** del modello. Attenzione e feedforward sono gli stessi identici pesi per qualunque lunghezza; l'unico tensore che dipende dal contesto è `pos_emb` (la tabella delle posizioni: 1.024×768 contro 256×768) più il buffer della maschera. Il modello resta "il 124M" a tutti gli effetti — è solo *miope*: non vedrà mai più indietro di 256 token. Per imparare lo stile di un racconto, bastano e avanzano.

**La conseguenza a valle**: questa scelta è esattamente il motivo per cui, nella sezione dei pesi OpenAI, nasce `NEW_CONFIG` con `context_length: 1024` (e `qkv_bias: True`) — i pesi originali hanno la tabella posizionale a 1.024 righe, e un modello costruito a 256 **non può riceverli** (forma di `pos_emb` incompatibile: te lo ha denunciato lo `state_dict` con le sue mani). Due configurazioni, due scopi: 256 per *allenare in piccolo il nostro*, 1.024 per *ospitare il loro*.

Infine, anche gpt-oss è stato addestrato su contesti più corti e poi *esteso* a 131.072 con YaRN. Ridurre (o crescere) il contesto per governare i costi del training non è un trucco da corso: è il mestiere.


<sup>1</sup> Lo stride è il passo di avanzamento della finestra scorrevole: di quanti token ci spostiamo tra un esempio e il successivo

Segue il consueto check delle versioni installate dei seguenti notebook.

Prima **installare da terminale `tensorflow`** con questi comandi:

```cmd
conda activate <env>
python -m pip install tensorflow
```

In [6]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow" # per i pesi pre-allenati di OpenAI
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.11.2
numpy version: 2.4.6
tiktoken version: 0.14.0
torch version: 2.14.1+cu130
tensorflow version: 2.21.0


# Valutare modelli di generazione di testo

Dopo una breve spiegazione della generazione del testo fatta dal notebook, configureremo il nostro LLM per la generazione del testo e poi discuteremo i metodi di base per valutare la qualità del testo generato. Calcoleremo quindi le perdite (*loss*) di allenamento e convalida.

La figura seguente mostra gli argomenti trattati in questo notebook, con i primi tre passaggi evidenziati.
![](figura_52_corretta.png)

## Usare GPT per generare testo

Configuriamo il modello LLM e riepiloghiamo brevemente il processo di generazione del testo implementato nel notebook 4. Iniziamo inizializzando il modello GPT che valuteremo e addestreremo successivamente, usando la classe `GPTModel` e il dizionario `GPT_CONFIG_124M` (vedi notebook 4):

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

Considerando il dizionario `GPT_CONFIG_124M`, **l’unica modifica** rispetto al notebook precedente riguarda la **riduzione della lunghezza del contesto** (*context_length*) a **256 token**. Questa modifica riduce il carico computazionale dell’addestramento del modello, rendendo possibile eseguire il training **su un normale computer portatile**.

Originariamente, il modello GPT-2 con 124 milioni di parametri era configurato per gestire fino a 1.024 token. Dopo l’addestramento, **aggiorneremo l’impostazione della *context length*  e caricheremo i pesi pre-addestrati per lavorare con un modello configurato per una lunghezza di contesto di 1.024 token**.

Ora useremo l’istanza di `GPTModel` e adotteremo perciò la funzione `generate_text_simple` dal notebook 4 e introdurremo **due funzioni utili**: `text_to_token_ids` e `token_ids_to_text`. Queste funzioni **facilitano la conversione** tra testo e rappresentazioni in token, tecnica che utilizzeremo per tutto il notebook.

La seguente figura spiega come la generazione di testo comporta la codifica del testo in token ID che il LLM elabora producendo **vettori di logit** (*logit vectors*). I vettori di logit vengono poi convertiti nuovamente in token ID e detokenizzati per ottenere una rappresentazione testuale.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-process.webp" width=800px>

Come si vede dalla figura, il processo di generazione del testo, quando si usa il modello GPT, è composto da **tre fasi**.

Per prima cosa, il tokenizer converte il testo di input in una serie di token ID (vedi notebook 2). In secondo luogo, il modello riceve questi ID e genera i corrispondenti logit, ovvero vettori che rappresentano la distribuzione di probabilità per ogni token nel vocabolario (vedi notebook 4). In terzo luogo, questi logit vengono convertiti nuovamente in token ID , che il tokenizer decodifica in testo leggibile, completando **il ciclo dall’input testuale all’output testuale**.

Possiamo **implementare** questo processo di generazione del testo, come mostrato nella seguente cella, che definisce anche **le due nuove funzioni di conversione** (di cui abbiamo detto prima):

In [7]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("testo di output:\n", token_ids_to_text(token_ids, tokenizer))

testo di output:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


`model` ha generato il testo:<br>
*Every effort moves you rentingetic wasnم refres RexMeCHicular stren*

Chiaramente, <u>il modello NON sta ancora producendo testo coerente</u> perché **non è stato addestrato**. Per definire cosa si intende per testo “coerente” o “di alta qualità”:
> dobbiamo implementare un **metodo numerico per valutare il contenuto generato**.
Questo approccio ci permetterà di **monitorare e migliorare** le prestazioni del modello durante tutto il processo di addestramento.

Successivamente, calcoleremo una metrica di perdita (*loss metric*) per gli output generati. Questa perdita servirà come **indicatore di progresso e successo del training**. Inoltre, nei notebook successivi, durante il *fine-tuning* del nostro LLM, esamineremo <u>metodologie aggiuntive per valutare la qualità del modello</u>.

## Calcolare la funzione di perdita della generazione del testo: cross-entropia e perplessità

Esploriamo quindi delle **tecniche** per:
> **valutare numericamente la qualità del testo generato durante l’addestramento calcolando una loss di generazione testuale**.
Tratteremo questo argomento <u>passo dopo passo con un esempio pratico</u>, per rendere i concetti chiari e applicabili, iniziando con un <u>breve riepilogo</u> di come vengono caricati i dati e di come viene generato il testo tramite la funzione `generate_text_simple`.

La seguente figura illustra il **flusso complessivo** dall’input testuale al testo generato dall’LLM, composto da **cinque fasi**.
> Questo processo mostra **ciò che fa internamente la funzione `generate_text_simple`**.

Dobbiamo eseguire **proprio** questi stessi passaggi iniziali prima di poter calcolare **una loss** che misuri la qualità del testo generato (vedi più avanti in questo notebook):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-to-text.webp" width=1000px>

Per ciascuno dei tre token di input mostrati a sinistra (*every*, *effort*, *moves*), la funzione `generate_text_simple` calcola un vettore contenente i **punteggi di probabilità** (*probability scores*) <u>per ogni token del vocabolario</u>. La posizione dell’indice con il punteggio più alto rappresenta il prossimo token più probabile. I token ID associati ai punteggi di probabilità più alti (per ogni token in input) vengono selezionati e mappati nuovamente in testo, che rappresenta il **testo generato dal modello**.

L'esempio in figura utilizza **un piccolo vocabolario di sette token**, per adattarsi alla figura! Tuttavia, il nostro `GPTModel` lavora con un vocabolario molto più grande, composto da **50.257 parole**; di conseguenza, i token ID nel codice vanno <u>da 0 a 50.256 invece che da 0 a 6</u>.

Inoltre, per semplicità, la figura mostra un **unico** esempio testuale ("every effort moves"). Nell’esempio successivo utilizziamo lo stesso processo della figura ma **con due esempi di input** al modello GPT ("every effort moves" e "I really like").

La seguente cella riporta i due esempi già mappati in token ID (il passo 1 della figura), ora con **vocabolario di 50.257 parole**.<br>
Il tensore `input` contiene i token ID dei **due esempi di training** (le due righe).<br>
In corrispondenza degli `input`, i `target` contengono token ID desiderati (cioè, che vogliamo che il modello generi).<br>
Si noti che i `target` sono gli `input` **spostati di 1 posizione**, come spiegato nel notebook 2 quando abbiamo implementato il data loader.

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

Questa strategia di *shifting* è <u>cruciale</u> per insegnare al modello a prevedere il token successivo in una sequenza.

Ora **inseriamo gli input nel modello** per **calcolare i vettori di logit per i due esempi**, ciascuno composto da tre token (le tre parole). Poi applichiamo la funzione `softmax` per trasformare questi logit in "punteggi di probabilità" (*probability scores*), le `probas` (è il passo 2 della figura di prima):

In [ ]:
with torch.no_grad():                  # disabilita il calcolo del gradiente perchè NON stiamo ancora allenando il modello
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # lo score di probabilità di ogni token del vocabolario
print(probas.shape)                    # shape: (batch_size, num_tokens, vocab_size)

Il primo numero della shape del tensore, 2, corrisponde ai due esempi (righe) negli input, anche detto **batch size**.
Il secondo numero, 3, corrisponde al **numero di token per ciascun input (la riga)**.<br>
Infine, l’ultimo numero corrisponde alla **dimensione dell’embedding**, determinata <u>dalla dimensione del vocabolario</u>.

Dopo la conversione da logit a probabilità tramite `softmax`, la funzione `generate_text_simple` - il cui processo interno stiamo appunto analizzando passo per passo - converte i punteggi di probabilità risultanti in testo (figura 5.4, passaggi 3–5).

Possiamo completare i passaggi 3 e 4 applicando la funzione `argmax` ai punteggi di probabilità per ottenere i token ID corrispondenti:

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

Dato che abbiamo due batch di input, ciascuno lungo tre token, l'applicazione della funzione `argmax` ai punteggi di probabilità (il passo 3 della figura) produce otteniamo **due output, ciascuno con i tre token ID PREVISTI ("generati")**, uno per il primo batch, l'altro per il secondo batch:

Infine, il passo 5 converte i token ID in testo:

In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Notiamo come questi token siano **molto diversi dai token target** da quelli che vogliamo il modello generi.

Il modello produce **testo casuale**, diverso dal testo atteso, perché **non è ancora stato addestrato**.

Vogliamo ora **valutare numericamente la performance del testo generato**, tramite una **loss**.

La metrica *loss* **misura "quanto distanti" siano i token generati rispetto alle previsioni corrette (*target*)**.

Questa metrica è utile per misurare la qualità del testo prodotto, ma è anche **fondamentale per implementare il training del modello**, cioè la procedura con la quale, <u>più avanti</u>, aggiorneremo i pesi del modello per migliorarne la capacità generativa. La funzione di training che implementeremo più avanti, infatti, userà queste stesse informazioni per "aggiustare/regolare" i pesi del modello, così da generare **testo sempre più simile (o, idealmente, identico) al testo *target***!

L’obiettivo dell’addestramento (*training*) è:
> **aumentare la probabilità *softmax* nelle posizioni degli indici corrispondenti ai token target corretti**,
come illustrato dalla seguente figura.

Questa probabilità *softmax* è appunto la **stessa** che useremo adesso come <u>metrica di valutazione</u> che stimi numericamente gli output generati dal modello: **più alta è la probabilità nella posizione corretta, meglio è**.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-index.webp" width=1000px>

Ricordiamo che, prima dell’addestramento, il modello produce vettori di probabilità per il token successivo <u>in modo casuale</u>, come abbiamo toccato con mano adesso. L’obiettivo del training è fare in modo che i valori di probabilità corrispondenti ai token target evidenziati vengano massimizzati.

Ricorda che la figura mostra le probabilità softmax per **un vocabolario ridotto di sette token**, in modo da far entrare tutto in una sola immagine. Questo implica che i valori casuali iniziali si aggireranno **intorno a 1/7, ovvero circa 0,14**. Tuttavia, il vocabolario che stiamo usando per il nostro modello GPT-2 contiene 50.257 token, quindi la maggior parte delle probabilità iniziali sarà circa **0,00002 (1/50.257)**.

Per ciascuno dei due testi di input, possiamo stampare i **punteggi di probabilità softmax iniziali** <u>corrispondenti ai token target</u>, utilizzando il seguente codice:

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("testo 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("testo 2:", target_probas_2)

Gli score sono appunto nell'ordine di **$10^{-5}$**.

L’obiettivo dell’addestramento di un LLM è **massimizzare la probabilità del token corretto**, il che significa <u>aumentare la sua probabilità rispetto a quella degli altri token</u>. In questo modo ci assicuriamo che l’LLM **selezioni sistematicamente il token target** — ossia la parola successiva nella frase — come prossimo token da generare.

**Nota sulla *Backpropagation***<br>
**Come si fa** a massimizzare le probabilità softmax corrispondenti ai token target?<br>
In sintesi, dobbiamo aggiornare i pesi del modello affinché l’output prodotto abbia valori più alti per token ID che vogliamo generare. L’aggiornamento dei pesi avviene attraverso un processo chiamato **backpropagation**, una tecnica standard per l’addestramento delle reti neurali profonde.

La *backpropagation* richiede una funzione di perdita (*loss function*), che calcola la differenza tra l’output predetto dal modello (cioè le probabilità associate ai token target) e l’output desiderato. **Questa loss misura quanto le previsioni del modello si discostano dai valori attesi**.

> **Nota matematica**<br>
> Vogliamo massimizzare tutti questi valori, portandoli vicini a una probabilità di 1.<br>
> Nell'ottimizzazione matematica è **più facile massimizzare il logaritmo del punteggio di probabilità** che il punteggio stesso; l'argomento esula da questo corso, ma una lezione con i dettagli è disponibile [qui](https://www.youtube.com/watch?v=GxJe0DZvydM).

Calcoliamo la *loss* per i punteggi di probabilità (*probability scores*) dei due batch di esempio: `target_probas_1` e `target_probas_2` (già calcolati prima). I passaggi principali sono illustrati nella seguente figura (i cui step 1-3 abbiamo già eseguito, come detto). Dunque procediamo con il passo 4: applicare il logaritmo ai punteggi di probabilità.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/cross-entropy.webp?123" width=1000px>

In [ ]:
# calcolo del logaritmo per le probabilità di tutti i token
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

Nella ottimizzazione matematica, lavorare con i logaritmi delle probabilità è più agevole rispetto all’uso diretto delle probabilità.

A seguire, combiniamo queste probabilità logaritmiche in un **unico punteggio calcolandone la media** (passaggio 5 nella figura):

In [ ]:
# calcolo della probabilità MEDIA per ogni token
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

L’**obiettivo è avvicinare il più possibile questo valore medio logaritmico allo 0** (che è il valore massimo con i log --> $scoremedio^0 = 1$),   aggiornando i pesi del modello durante il training. Attualmente siamo lontani da questo valore!

Tuttavia, nel deep learning, la pratica comune non è portare la log-probabilità media a 0, ma piuttosto **ridurre a 0 la log-probabilità media negativa**. Quest’ultima è semplicemente la **log-probabilità media moltiplicata per –1**, come indicato nel passaggio 6 della figura.

In [ ]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

Nel nostro esempio, anzichè massimizzare -10.7940 così che avvicini 0, preferiamo - secondo la prassi del Deep Learning - minimizzare 10.7940 così che si avvicini a 0.

Nel deep learning, questo valore — da negativo (-10.7940) a positivo (10.7940) — è noto come **cross entropy loss**. PyTorch ci viene in aiuto, poiché include già una funzione `cross_entropy` che si occupa internamente di **tutti e sei i passaggi illustrati** nella precedente figura.

**Cross entropy loss**<br>
La [*cross entropy loss*](https://en.wikipedia.org/wiki/Cross-entropy) è una **misura** molto usata nel machine learning e nel deep learning per **quantificare la distanza tra due distribuzioni di probabilità** — tipicamente, la distribuzione **reale** delle etichette (in questo caso, i token in un dataset) e quella **prevista** dal modello (cioè le probabilità dei token generate da un LLM).<br>
Nel contesto del machine learning e di PyTorch, la funzione `cross_entropy` calcola questa misura per **risultati discreti**, ed è matematicamente equivalente alla log-probabilità media negativa dei token target, data la distribuzione prodotta dal modello. Per questo motivo, i termini “cross entropy” e “log-probabilità media negativa” sono correlati e <u>spesso usati in modo intercambiabile</u>.

Prima di applicare la funzione `cross_entropy`, ricordiamo brevemente le *shape* dei tensori `logits` e `targets`:

In [ ]:
# i logit hanno shape (batch_size, num_tokens, vocab_size)
print("shape dei logits:", logits.shape)

# i target hanno have shape (batch_size, num_tokens)
print("shape dei target:", targets.shape)

Come possiamo vedere, il tensore `logits` ha tre dimensioni: dimensione del batch, numero di token, e dimensione del vocabolario. Il tensore `targets` ha due dimensioni: dimensione del batch e numero di token.

Per usare la funzione `cross_entropy` di PyTorch, dobbiamo appiattire (*flatten*) questi tensori combinandoli sulla dimensione del batch:

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("logit appiattiti:", logits_flat.shape)
print("target appiattiti:", targets_flat.shape)

Ricorda che i targets sono i token ID che vogliamo che l’LLM generi, mentre i logit contengono gli output del modello non scalati, prima di essere trasformati in punteggi di probabilità tramite la `softmax`.

In precedenza abbiamo applicato la funzione `softmax`, selezionato i punteggi associati ai target, e calcolato la log-probabilità media negativa.

La funzione `cross_entropy` di PyTorch svolge per noi **automaticamente** tutti questi passaggi:

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

La loss risultante è **la stessa** che avevamo calcolato <u>manualmente</u> con i singoli passaggi della figura.

## La *perplexity*
La perplexity è una **misura**, <u>spesso usata insieme alla cross entropy loss</u>, per **valutare le prestazioni dei modelli in compiti come il language modeling**. Dal punto di vista matematico, essa è semplicemente **l'esponenziale della cross-entropy loss**.
    
Può fornire un **modo più interpretabile** (rispetto alla *loss*) per comprendere <u>l’incertezza</u> di un modello nel prevedere il token successivo in una sequenza.

La perplexity misura **quanto bene la distribuzione di probabilità *prevista* dal modello corrisponda alla distribuzione *reale* delle parole nel dataset**.
> Come per la loss, una perplexity <u>più bassa</u> indica che le previsioni del modello sono <u>più vicine alla distribuzione effettiva</u>.

La perplexity può essere <u>calcolata</u> come: **perplexity = `torch.exp(loss)`**, che restituisce **tensor(48725.8203)** se applicato alla loss calcolata in precedenza.

La perplexity è spesso considerata **più interpretabile del valore grezzo della loss**, come detto, perché indica la dimensione (*size*!) effettiva del vocabolario su cui il modello è **incerto a ogni passaggio**.
    
Nel nostro esempio, perplexity = 48725 vuol dire che:
> il modello è **incerto su quale dei 48.725 token presenti nel vocabolario scegliere come prossimo token**.

In [ ]:
perplexity = torch.exp(loss)
print(perplexity)

Nelle celle precedenti abbiamo calcolato la *loss* per **due piccoli input testuali**, a scopo illustrativo.<br>
Il passo successivo è **applicare il calcolo della loss all’intero set di training e validazione (test)**.

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>E in gpt-oss? — La loss è la stessa, fino all'ultima formula</b><br><br>
Il pre-training di gpt-oss minimizza <u>esattamente</u> la cross-entropia sul prossimo token appena costruita: softmax → probabilità dei target → logaritmo → media → negazione. Nessuna variante; e anche la perplexity resta la metrica di riferimento.<br><br>
L'unica aggiunta è figlia del MoE visto nel notebook 4: come è prassi nell'allenamento dei Mixture-of-Experts, alla loss principale si affianca una piccola <b>loss ausiliaria di bilanciamento del carico</b> (<i>load balancing</i>), che penalizza il router quando concentra i token su pochi esperti — senza di essa, alcuni esperti finirebbero per non essere mai usati. Qualche riga in più nella funzione di perdita, non un'altra matematica.
</div>

## Preparare il training e test dataset e calcolare le loss su di essi

Dobbiamo prima **preparare i dataset di training e validazione** che useremo per addestrare il nostro LLM.

Poi, come mostrato nella figura successiva, calcoleremo la **cross entropy per entrambi i dataset**, un passaggio <u>fondamentale</u> nel processo di training del modello.

![](Figure_5_8.png)

Dopo aver completato i passaggi 1 e 2, incluso il calcolo della cross entropy loss, possiamo ora **applicare questo calcolo all’intero dataset testuale** che useremo per addestrare il modello.

Per calcolare la loss sui dataset di training e validation, utilizziamo un **dataset testuale molto piccolo**: il breve racconto **“The Verdict” di Edith Wharton**, con cui abbiamo già lavorato nel notebook 2.

Scegliendo un testo di *Wikisource*, evitiamo qualsiasi problema legale legato ai diritti d’uso. Inoltre, usare un dataset così piccolo consente di eseguire gli esempi di codice **su un normale portatile nel giro di pochi minuti**, anche senza GPU di fascia alta — un vantaggio importante in ambito didattico.

**📌 Nota**<br>
Il corso rende disponibile il **codice supplementare** per preparare un **dataset su larga scala**, contenente oltre **60.000 libri** di pubblico dominio da [Project Gutenberg](https://it.wikipedia.org/wiki/Progetto_Gutenberg), ed addestrare un LLM su questi testi.

**Il costo del pretraining degli LLM**<br>
Per avere un’idea della scala di un progetto reale, consideriamo l’addestramento del **[modello LLaMA 2](https://it.wikipedia.org/wiki/Llama_(modello_linguistico)), con 7 miliardi di parametri**, un LLM abbastanza popolare e open source.<br>
Questo modello ha richiesto **184.320 ore GPU** su costose **GPU A100**, processando <u>2 trilioni di token</u>. A fine 2024 l'esecuzione su un<u> server cloud con 8 GPU A100 su AWS</u> costa circa **30 dollari all’ora**.<br>
Una stima approssimativa fissa il **costo complessivo dell’addestramento di tale LLM a circa 690.000 dollari** (calcolati come 184.320 ore ÷ 8 × 30 dollari).

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>E in gpt-oss? — La scala dei dati (e del contesto)</b><br><br>
La nota qui sopra dà i numeri di LLaMA 2; per gpt-oss l'ordine di grandezza è quello: un addestramento su <b>trilioni di token</b> — testo prevalentemente in inglese, con focus su STEM, programmazione e conoscenza generale — contro i <b>~20.000 token</b> scarsi del nostro racconto. OpenAI dichiara circa <b>2,1 milioni di ore-H100</b> (la H100 è la GPU NVIDIA da datacenter, 80 GB di memoria) per il 120b, e circa dieci volte meno per il 20b.<br><br>
A quelle scale il dataset si attraversa <u>più o meno una volta sola</u>: ecco il motivo, già anticipato nell'innesto del notebook 3, per cui il dropout è andato in pensione.<br><br>
Anche il <b>contesto</b> cresce per fasi: noi, per allenare in piccolo, lo riduciamo a 256 token; gpt-oss è stato addestrato e poi <u>esteso</u> fino a 131.072 token con YaRN (innesto del notebook 2).
</div>

Il codice seguente carica il racconto breve *“The Verdict”*:

In [ ]:
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

I primi 100 caratteri, come verifica:

In [ ]:
print(text_data[:99])

Gli ultimi 100 caratteri, come verifica:

In [ ]:
print(text_data[-99:])

Dopo aver caricato il dataset, possiamo controllare il **numero di caratteri e token del testo**:

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("caratteri:", total_characters)
print("token:", total_tokens)

Con solo 5.145 token, il testo può **sembrare troppo piccolo** per addestrare un LLM, ma come già detto, è a scopo educativo, per poter eseguire il codice **in pochi minuti invece che settimane**, e su un portatile non di ultima generazione (circa 2020). Comunque, a fine notebook, caricheremo i **pesi pre-addestrati di OpenAI nel nostro modello `GPTModel`**.

Ora, **dividiamo** il dataset in un training set e un validation set, e usiamo i **data loader** del notebook 2 per **preparare i batch per l’addestramento dell’LLM**. Questo processo è visualizzato nella seguente figura.

Per motivi di spazio, nella figura usiamo `max_length=6` token. Tuttavia, per i data loader **reali**, impostiamo `max_length=256` token — la lunghezza del contesto supportata dall’LLM — in modo che veda testi più lunghi durante l’addestramento.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/batching.webp" width=800px>

Come si vede in figura, <u>durante la preparazione dei data loader</u>, 1. **suddividiamo** il testo di input in porzioni per il training e la validazione. Poi 2. **tokenizziamo il testo** (in figura mostrato solo per il training set, per semplicità) e 3. lo **dividiamo** in blocchi (*chunk*) della lunghezza desiderata dall'utente (qui 6). Infine, 4. rimescoliamo (*shuffle*) le righe e organizziamo i blocchi di dati (*chunked text*) in batch (qui `batch size = 2`), che useremo per il training del modello.<br>
La figura NON mostra, per semplicità, i token target. Sappiamo che, in un modello di previsione della parola successiva, i token target sono gli stessi di quelli di input, **eccetto che sono shiftati di 1 posizione**.

**📌 Nota**<br>
Stiamo addestrando il modello con blocchi di testo di **dimensione simile**, per semplicità ed efficienza. Tuttavia, nella pratica, può essere utile addestrare un LLM anche **con input di lunghezza variabile**, per aiutarlo a generalizzare meglio a diverse tipologie di testo.

Per implementare la suddivisione e il caricamento dei dati, definiamo prima una variabile `train_ratio` per usare il **90% dei dati per il training e il restante 10% per la validazione**:

In [ ]:
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

Usando i subset `train_data` e `val_data`, possiamo ora creare i **rispettivi data loader** riutilizzando la funzione `create_dataloader_v1` dal notebook 2 (e reimportata qui ad inizio notebook):

In [ ]:
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

Abbiamo usato un *batch size* relativamente piccolo **(2)** per ridurre il carico computazionale, dato che stiamo lavorando con un dataset molto ridotto.<br>
Nella pratica, non è raro addestrare LLM con `batch size` **1.024** o più.

Un paio di *sanity check*:

In [ ]:
if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

Come controllo opzionale, possiamo **iterare sui data loader** per assicurarci che siano stati creati correttamente:

In [ ]:
print("train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nvalidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Un'altra verifica (opzionale):  le dimensioni dei token (*token sizes*) sono quelle previste?

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("token di training tokens:", train_tokens)
print("token di validazione:", val_tokens)
print("token complessivi:", train_tokens + val_tokens)

La creazione dei data loader ha prodotto **nove batch di training set**, ciascuno con due esempi da 256 token. Poiché abbiamo assegnato solo il 10% dei dati alla validazione, esiste **un solo batch di validation** con due esempi di input.

Come previsto, i dati di input (`x`) e i dati target (`y`) hanno la stessa *shape* (batch size × numero di token), poiché i target sono semplicemente gli input spostati di una posizione, come sappiamo dal notebook 2.

Ora implementiamo una **funzione di utilità** per calcolare la *cross entropy loss* di un batch restituito dai data loader (di training e di validazione):

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):                   # 'device' permette di trasferire i dati
                                                                                 # su GPU
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

Possiamo usare questa funzione `calc_loss_batch`, che calcola la loss **su un singolo batch**, per implementare la funzione `calc_loss_loader`, che calcola la loss su **tutti i batch campionati da un determinato data loader**, dove il numero di batch è specificato dall'utente in `num_batches`.

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)    # itera su tutti i batch se non viene specificato alcun 'num_batches' fisso
    else:
        # Riduce il numero di batch per farlo corrispondere al numero totale di batch nel data loader
        # se 'num_batches' supera il numero di batch nel caricatore dati.
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()     # somma la loss per ogni batch
        else:
            break
    return total_loss / num_batches

Per default, la funzione `calc_loss_loader` **itera su tutti i batch** presenti in un data loader, **accumula la loss** in `total_loss` e poi **calcola la loss media** sul numero totale di batch. <u>In alternativa</u>, possiamo specificare un numero minore di batch con `num_batches`, per **velocizzare la valutazione durante il training**.

Se il PC dispone di una **GPU con supporto CUDA**, l'LLM si allenerà sulla GPU <u>senza</u> apportare modifiche al codice.<br>
Tramite infatti l'impostazione `device`, ci assicuriamo che i **dati vengano caricati sullo stesso dispositivo del modello LLM**.

Vediamo ora `calc_loss_loader` in azione, applicandola ai loader di training e validation.<br>
Prima bisogna impostare il `device`:
* in **Jupyter Notebook**, l'istruzione della cella succesiva imposta correttamente il device (a seconda che CUDA sia rilevato)
* in Google Colab, dipende dal runtime impostato (dalla voce di menù: *change runtime type*):
    - se CPU oppure altri --> l'istruzione seguente imposta device CPU
    - se T4 GPU oppure altri --> l'istruzione seguente imposta device GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

> 📌 **Anche la versione di PyTorch con supporto CUDA può eseguire i calcoli sulla CPU**. È sufficiente sostituire la selezione automatica con:
>
> ```cmd
> device = torch.device("cpu")
> print(device)  # stampa: cpu
> ```
>
> `device` è una normale variabile Python, che contiene il dispositivo da usare nelle istruzioni successive. L’assegnazione non modifica una configurazione permanente di PyTorch o dell’ambiente Conda.<br>
> Lo stato del notebook appartiene al kernel, cioè al processo che esegue il codice (tuttavia, chiudere soltanto la scheda del browser può lasciare attivo il kernel e quindi conservare le variabili.]<br>
> In uno script *.py*, l’assegnazione vale nel suo normale ambito durante quell’esecuzione; alla successiva esecuzione viene eseguita nuovamente.
>
> La scelta va applicata a modello e dati:<br>
> Per eseguire sulla CPU un modello `model` con un tensore di input x, già creati:<br>
> ```python
>   device = torch.device("cpu")
>   model = model.to(device)  # Colloca il modello sulla CPU
>   x = x.to(device)          # Colloca l'input x (il tensore) sulla CPU
>   output = model(x)
> ```
> `.to(device)` applica la destinazione <u>al modello e al tensore</u>. Cambiare soltanto il valore della variabile `device` lascia gli oggetti già esistenti sul dispositivo in cui si trovano. [PyTorch 2.14 documentation]. Per esempio, se il modello era già sulla GPU, assegnare `device = torch.device("cpu")` richiede poi di eseguire anche `model.to(device)` per trasferirlo.

👉 La seguente cella **può dare errore** (dipende da quale cuXXX di pytorch era stata installata: `cu126` o `cu130`): è il classico "battesimo" della RTX 5090!<br>

In [ ]:
# Nota:
# Rimuovendo il commento dalle seguenti righe sottostanti, il codice potrà essere eseguito sui chip Apple Silicon, se
# disponibili su MacOS, che sono circa 2 volte più veloci rispetto a una CPU Apple (come misurato su un MacBook Air M3).
# Tuttavia, i valori di loss risultanti potrebbero essere leggermente diversi.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")


model.to(device)         # l'assegnazione model = model.to(device) NON è necessaria per le classi 'nn.Module'

torch.manual_seed(123)   # per garantire la riproducibilità dei risultati (c'è un rimescolamento casuale nel data loader)

with torch.no_grad():    # disabilita il tracciamento del gradiente (cioè la costruzione del grafo computazionale), per efficienza, poichè NON stiamo ancora allenando
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("loss di training:", train_loss)
print("loss di validazione:", val_loss)

---
Il PyTorch installato è compilato per GPU fino alla generazione precedente, e la RTX 5090 non è tra quelle. Decifriamo il messaggio: ogni GPU NVIDIA ha una *compute capability (CC)*, il "numero di generazione" del suo set di istruzioni — la 5090 (architettura Blackwell) è **`sm_120`**, cioè **CC 12.0**. I pacchetti di PyTorch contengono kernel CUDA precompilati <u>solo per un elenco di CC</u>, e la build `torch==2.14.1+cu126` prima installata arriva fino a `sm_90`: <u>la scheda viene vista</u> (per questo il warning e non un errore secco), ma alla prima operazione vera su GPU si otterrebbe un errore tipo "no kernel image is available for execution on the device" — **in pratica, con questa installazione pytorch i notebook su CUDA non girano!!**

La cura è quella che il warning stesso suggerisce: stessa versione di *torch*, wheel costruito per un CUDA più recente, che **include `sm_120`**.

```cmd
conda activate <env>
python -m pip list | findstr torch
```

Il secondo comando ci dice se nell'ambiente ci sono anche `torchvision`/`torchaudio`: **se ci sono, vanno reinstallati insieme a *torch* dallo stesso indice, altrimenti resterebbero legati alla vecchia build:

```cmd
python -m pip uninstall torch torchvision torchaudio   # dipende da cosa era stato installato oltre a torch; anche torchaudio e/o torchvision
python -m pip install torch==2.14.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
```

Tra i tre indici proposti:

```text
CUDA 13.0 → https://download.pytorch.org/whl/cu130
CUDA 13.2 → https://download.pytorch.org/whl/cu132
CUDA 13.4 → https://download.pytorch.org/whl/cu134
```
 **si consiglia cu130**: è il più conservativo sul fronte driver — il wheel si porta dentro il runtime CUDA che gli serve, l'unico requisito esterno è che il driver NVIDIA sia abbastanza recente, e per CUDA 13.0 lo è qualunque driver che già pilota una 5090; cu132/cu134 vanno ugualmente bene se il driver è aggiornatissimo, ma non portano alcun vantaggio per i nostri notebook.


> 👉 **poi fare *restart kernel*!**

 ---

I valori di loss sono <u>piuttosto alti</u>, perché il **modello non è ancora stato addestrato**.<br>
A titolo di confronto, la loss tende a zero quando il modello **impara a generare correttamente i token successivi** come appaiono nei set di training e validation.

Ora che abbiamo un **modo per misurare la qualità del testo generato**, procederemo con l’addestramento dell’LLM per **ridurre questa loss**, migliorando così la generazione di testo. La seguente figura fa un punto della situazione. Il passo successivo sarà definire le funzioni di training e pre-addestrare il nostro LLM (passo 4).

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-1.webp" width=600px>

# Allenare un LLM

**È finalmente arrivato il momento di implementare il codice per il pre-addestramento del nostro LLM**, il modello `GPTModel`! Ci concentreremo su un <u>ciclo di training semplice</u>, per mantenere il codice conciso e leggibile.

**📌 Nota**<br>
Chi fosse interessato ad approfondire può consultare altri testi, dove sono trattate tecniche più avanzate, tra cui il *learning rate warmup*, il *cosine annealing* e il *gradient clipping*.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/train-steps.webp" width=600px>

La figura descrive un **tipico ciclo di training** per reti neurali profonde in PyTorch, che è composto da **numerosi passaggi**, che **iterano** sui batch del training set per diverse *epoche*.<br>
In ogni ciclo, calcoliamo la *loss* per ciascun batch e poi i **gradienti della loss**, che usiamo per **aggiornare i pesi del modello**, minimizzando così la loss del training set.

Il diagramma di flusso in figura descrive proprio questo flusso di lavoro "standard in PyTorch", che utilizziamo anche per l’addestramento di un LLM.<br>
Comprende **otto passaggi**: si inizia iterando sulle epoche, processando i batch, azzerando i gradienti, calcolando la loss e i nuovi gradienti, aggiornando i pesi, e concludendo con attività di monitoraggio, come la visualizzazione delle perdite e la generazione di esempi di testo.

Possiamo implementare questo flusso di addestramento con la funzione `train_model_simple`.

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # inizializza le liste per tracciare le losses e i token visti
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # CICLO di training PRINCIPALE
    for epoch in range(num_epochs):
        model.train()  # imposta il modello in modalità TRAININGe

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # reset dei gradienti di loss dalle precedenti iterazioni batch
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()       # calcola i gradienti di loss
            optimizer.step()      # modifica i pesi del modello tramite i gradienti di loss
            tokens_seen += input_batch.numel()
            global_step += 1

            # step di valutazione (opzionale)
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # stampa un testo di esempio dopo ogni epoca
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

Da notare che la funzione `train_model_simple` ora creata utilizza **due funzioni di supporto** che non abbiamo ancora definito: `evaluate_model` e `generate_and_print_sample`.

La funzione `evaluate_model` corrisponde al passo 7 della figura precedente. Serve a stampare i valori della loss per i set di training e validation <u>dopo ogni aggiornamento del modello</u>, per **capire se il training sta effettivamente migliorando le prestazioni**.<br>
In particolare, `evaluate_model` calcola la loss su training e validation set, assicurandosi che il **modello sia in modalità di valutazione (evaluation mode)**, con il *dropout* ed il *tracciamento dei gradienti* **disabilitati**.

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()            # durante la valutazione il dropout è disabilitato per garantire
                            # risultati stabili e riproducibili

    with torch.no_grad():   # disabilità il "gradinet tracking", che NON è richiesto durante la valutazione, per
                            # ridurre il peso computazionale

        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()           # ritorna alla modalità train
    return train_loss, val_loss

Analogamente, la funzione `generate_and_print_sample` è comoda per **verificare** se il **modello sta migliorando nel corso del training**. Riceve in input una porzione di testo (*start_context*), la converte in token ID, e la passa all'LLM per generare un testo, usando la funzione `generate_text_simple` vista in precedenza.

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # compatta il formato di stampa
    model.train()

Mentre `evaluate_model` fornisce una **misura numerica del progresso**, `generate_and_print_sample` fornisce un **esempio concreto di testo generato**, utile per valutare le capacità del modello.

**Nota su *AdamW***<br>
Gli ottimizzatori *Adam* sono una **scelta popolare** per addestrare reti neurali profonde. Sono descritti in [questo breve video](https://www.tiktok.com/@brainlink_project/video/7557364249153113366) di *brainlink*, una ottima academy di AI e ML. Tuttavia, nel nostro ciclo di training <u>utilizzeremo l'ottimizzatore *AdamW*</u>.<br>
AdamW è una **variante** di Adam che migliora la gestione della penalizzazione dei pesi (*weight decay*), riducendo la complessità del modello e **prevenendo l’overfitting**.<br>
Questo approccio consente una **regolarizzazione più efficace e una migliore generalizzazione**, motivo per cui AdamW è <u>spesso impiegato</u> nell’addestramento degli LLM.

**Vediamo ora tutto questo in pratica**: addestriamo un’istanza di `GPTModel` per **10 epoche** usando **AdamW** e la funzione `train_model_simple` definita in precedenza.<br>
L'esecuzione di `train_model_simple` avvia il processo di training, che richiede circa:
* **11 minuti** su un desktop Windows con GPU NVIDIA CUDA (2GB RAM) - AP
* **5 minuti** su un MacBook Air (Raschka ed altri)
* **56 secondi** su un T4 di Google Colab - AP
* **0.23 minuti** su GPU RTX 5090 - AP

In [ ]:
# Nota:
# scommentare il codice seguente se si vuole calcolare il tempo di esecuzione
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),            # il metodo 'parameters' restituisce tutti i pesi allenabili del modello
    lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(    # <-- avvia il training
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Nota:
# scommentare il codice seguente se si vuole calcolare il tempo di esecuzione
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"training completato in {execution_time_minutes:.2f} minuti.")

Come possiamo vedere, la **loss del training migliora notevolmente**: parte da un valore di 9.824 e converge verso 0.9141.
Le capacità linguistiche del modello sono migliorate significativamente.

All’inizio, il modello riusciva solo ad **aggiungere virgole al contesto iniziale** ("Every effort moves you,,,,,,,,,,,") o a **ripetere la parola *and*.**

Alla fine del training, è in grado di **generare testo grammaticalmente corretto**.

Anche la validation loss segue <u>un andamento simile</u>: parte da un valore alto (9.929 e diminuisce durante l’addestramento. Tuttavia, **non scende mai quanto la training loss**, e si assesta a 6.317 dopo la decima epoca.

E' possibile ottenere valori di loss **leggermente diversi** su ogni PC, il che non è motivo di preoccupazione se sono più o meno simili (una loss di training inferiore a 1 e una loss di validazione inferiore a 7).

Piccole differenze possono spesso dipendere da GPU differenti, versioni di CUDA differenti o da piccole modifiche introdotte nelle versioni più recenti di PyTorch.

Anche se stai eseguendo l’esempio su CPU, potresti osservare leggere differenze; una possibile causa di discrepanze è il comportamento diverso di `nn.Dropout` tra sistemi operativi, a seconda di come PyTorch è stato compilato, come discusso [qui, nel tracker dei problemi di PyTorch](https://github.com/pytorch/pytorch/issues/121595).

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>E in gpt-oss? — Chi può allenare che cosa</b><br><br>
Il nostro 124M si pre-allena in minuti su una GPU qualsiasi: lo abbiamo appena fatto. Per gpt-oss, la scala vista sopra chiude la porta: il pre-training da zero non è replicabile da nessun singolo. Che cosa resta possibile su hardware comune?<br><br>
<table style="border-collapse:collapse">
<tr><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">Operazione</th><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">GPT-2 124M (il nostro)</th><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">gpt-oss-20b</th><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">gpt-oss-120b</th></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Pre-training da zero</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✔ minuti</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✘</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✘</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Fine-tuning completo</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✔ (notebook 6-7)</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✘</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✘</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Fine-tuning LoRA / QLoRA</td><td style="border:1px solid #9bb8d3; padding:6px 10px">—</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✔ (QLoRA: ~14 GB di VRAM)</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✘ (~65 GB)</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Inferenza</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✔</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✔ (~13 GB in MXFP4)</td><td style="border:1px solid #9bb8d3; padding:6px 10px">✔ con offload degli esperti in RAM</td></tr>
</table>
<br>
Tre termini nuovi: <b>LoRA</b> (<i>Low-Rank Adaptation</i>) congela i pesi del modello e allena solo piccole matrici "adattatrici" a basso rango accanto a essi; <b>QLoRA</b> è LoRA con il modello base quantizzato a 4 bit; <b>offload</b> significa che parte dei pesi vive nella RAM di sistema ed è servita dalla CPU. Attenzione inoltre: MXFP4 <u>non è un formato allenabile</u> — per il training i pesi vanno riportati in bf16, ed è lì che i requisiti esplodono.<br><br>
La buona notizia didattica: nel modulo dedicato pre-alleneremo comunque "da zero" un <b>mini-gpt-oss giocattolo</b> (stessa architettura, pochi milioni di parametri) proprio con il <code>train_model_simple</code> di questo capitolo. La pedagogia sopravvive intatta.
</div>

Prima di analizzare più nel dettaglio la validation loss, creiamo un **semplice grafico** che confronta training e validation loss nel tempo.

---
📌 **NOTA — Se il kernel "muore" sulla prima cella matplotlib (Windows): il conflitto OpenMP**

Su alcune macchine Windows, la **prima cella che usa *matplotlib* dopo *torch*** termina il kernel all'istante, senza alcun errore Python. Attenzione: il crash non appartiene a una cella precisa — colpisce la *prima* cella matplotlib eseguita nella sessione, quindi può presentarsi qui (grafico delle loss) o più avanti, a seconda dell'ordine di esecuzione. Su Google Colab il problema non esiste.

**La causa**: nel processo vengono caricate <u>due copie</u> del runtime **OpenMP** (la libreria di calcolo parallelo multi-thread) — una portata da PyTorch installato con pip, l'altra dalla MKL (la *Math Kernel Library* di Intel) che accompagna numpy/matplotlib installati con conda. Su Windows la seconda copia fa abortire il processo.

**La diagnosi** (dal prompt, con l'ambiente attivo — in terminale l'errore si vede, nel notebook no):

```
python -c "import torch; import matplotlib.pyplot as plt; plt.plot([1,2,3]); plt.savefig('t.png'); print('ok')"
```

Se compare `OMP: Error #15: Initializing libiomp5md.dll...`, è questo il problema.

**La cura risolutiva** — eliminare il doppione, riportando numpy e matplotlib alla filiera pip (stesse versioni, cambia solo la variante di build: da MKL a OpenBLAS). Leggiamo le versioni installate:

```
python -c "import numpy, matplotlib; print(numpy.__version__, matplotlib.__version__)"
```

e reinstalliamo quelle identiche:

```
python -m pip install --force-reinstall numpy==<versione> matplotlib==<versione>
```

> 📌 **Unica regola di convivenza, d'ora in poi: in questo ambiente *numpy* e *matplotlib* non vanno più toccati con `conda install`/`conda update`. Verifica finale: il comando di diagnosi di prima deve stampare `ok`. Poi riavviamo il kernel.**

**Il palliativo** (rapido ma non garantito): la variabile `KMP_DUPLICATE_LIB_OK=TRUE`, che chiede a OpenMP di tollerare il doppione — come primissima cella del notebook, prima di ogni import:

```python
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
```

Onestà dovuta: Intel la classifica come scappatoia non supportata, e nella nostra esperienza diretta (VS Code su Windows) **non è bastata** — il kernel moriva comunque. Vale come primo soccorso; la cura vera è la reinstallazione qui sopra.

---

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

![](Figure_5_12_corretta.png)

Come possiamo vedere, entrambe le perdite diminuiscono all’inizio, segno che il modello sta imparando.<br>
Tuttavia, dopo la seconda epoca, le curve **iniziano a divergere**. Questa divergenza, unita al fatto che la validation loss è molto più alta della training loss, indica che il modello sta **andando in overfitting sui dati di training**.

Possiamo verificare che il modello **memorizza <u>letteralmente</u> (*verbatim*) i dati di training** cercando nel file “The Verdict” frasi del testo generato come, ad esempio, "quite insensible to the irony", che si trovano .

Questa memorizzazione esatta è prevedibile, visto che stiamo lavorando con un **dataset molto, molto piccolo** e stiamo addestrando il modello con diverse epoche.<br>
Di solito, è più comune addestrare un modello su un dataset molto più grande e con meno epoche (epr questioni computazionali).

**👉 Nota**<br>
Come accennato, chi è interessato, può provare ad addestrare il modello su 60.000 libri di pubblico dominio presi da Project Gutenberg, dove questo tipo di overfitting non si verifica.

> 📌 **NOTA D'AULA — Il momento "Wharton"**<br>
> Si noti il **prima/dopo** della stessa funzione `generate_and_print_sample`: a inizio training produce parole a caso; dieci epoche dopo genera frasi nello stile di Edith Wharton, l'autrice di *"The Verdict"*. Con un dataset così piccolo, in realtà, il modello **memorizza** e ricita interi passaggi quasi alla lettera: è l'overfitting appena diagnosticato — che qui, da difetto, diventa dimostrazione visibile che i pesi hanno davvero assorbito il testo.

**Punto della situazione**<br>
Il nostro modello `GPTModel`, dopo il training, è ora in grado di **generare testo coerente**.<br>
Tuttavia, tende ancora a **memorizzare i passaggi** del training set.<br>
A questo punto, vediamo alcune **strategie** per generare **testi più vari e ridurre la memorizzazione esatta**.
Facciamo un punto della situazione del corso:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-2.webp" width=700px>

Come illustrato nella figura, abbiamo completato quattro dei nostri obiettivi per questo notebook.
Nel prossimo capitolo vedremo alcune strategie di generazione del testo per LLM, utili a ridurre la memorizzazione dei dati di training e **aumentare l’originalità del testo prodotto**, prima di passare al <u>caricamento e salvataggio</u> dei **pesi pre-addestrati del modello GPT di OpenAI**.

# Le "strategie di decoding" per controllare la casualità

Esploriamo ora le strategie di generazione del testo (note anche come *strategie di decoding*) per produrre **testo più originale**.

Iniziamo riprendendo brevemente la funzione `generate_text_simple`, che avevamo usato all’interno della `generate_and_print_sample`.

Poi introdurremo due <u>tecniche</u> per migliorare questa funzione: il **temperature scaling** e il **top-k sampling**.

Cominciamo riportando il modello **da GPU a CPU** - l’inferenza con un modello relativamente piccolo come questo **non richiede una GPU**.

Inoltre, dopo il training, mettiamo <u>il modello in modalità di valutazione</u> (*evaluation mode*) per disattivare le componenti casuali come il dropout:

In [ ]:
model.to("cpu")
model.eval()

Successivamente, colleghiamo l’istanza di `GPTModel` alla funzione `generate_text_simple`, che utilizza l’LLM per generare un token alla volta:

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Come spiegato in precedenza, a ogni passo di generazione **il token generato è quello con il punteggio di probabilità più alto tra tutti quelli nel vocabolario**.
Questo significa che l’LLM:
> genererà **sempre lo stesso output**, anche se eseguiamo più volte la funzione `generate_text_simple` sullo <u>stesso contesto iniziale</u> ("Every effort moves you").

## Lo *scaling* della temperatura

Vediamo ora la tecnica del *temperature scaling*, che introduce un aspetto **elemento probabilistico** nella selezione del token successivo.

In precedenza, nella funzione `generate_text_simple`, usavamo la funzione `torch.argmax` (una strategia detta *greedy decoding*), per cui selezionavamo **sempre** il token con la probabilità più alta.

Per ottenere una <u>maggiore varietà nel testo</u>, possiamo
> **sostituire `argmax`** con una **funzione che CAMPIONA i token da una distribuzione di probabilità** (in questo caso, i punteggi generati dall’LLM per ogni voce del vocabolario a ogni passo).

Per illustrare questo **sampling probabilistico**, utilizziamo un esempio concreto con un <u>vocabolario molto piccolo</u>, il seguente:

In [ ]:
# il piccolo vocabolario text token --> ID token
vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
}

# il piccolo vocabolario inverso : ID token --> text token
inverse_vocab = {v: k for k, v in vocab.items()}

print("vocab: ",vocab)
print("\n vocab inverso: ",inverse_vocab)

Supponiamo che l’LLM riceva come **contesto iniziale** il solito "every effort moves you" e produca i **seguenti logit** per il token successivo:

In [ ]:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)
next_token_logits

Come si vede ad occhio, i logit più alto è il quarto (6.75) corrispondente a "forward".

Come sappiamo dal notebook 4, all’interno di `generate_text_simple` **convertiamo i logit in probabilità con la funzione softmax** e otteniamo **l’ID del token** generato (più probabile) con `argmax`, che poi possiamo convertire in testo <u>usando il vocabolario inverso</u>.

In [ ]:
probas = torch.softmax(next_token_logits, dim=0)    # conversione dei logit in probabilità
next_token_id = torch.argmax(probas).item()         # l'ID del token più probabili

# il token generato (testuale, grazie al vocabolario inverso):
print(inverse_vocab[next_token_id])

Il valore logit più alto, e di conseguenza lo score di probabilità softmax maggiore, è in quarta posizione (indice 3, considerando l’indicizzazione da zero di Python), il token generato è "forward".

Per implementare un **campionamento probabilistico** (*probabilistic  sampling*), dobbiamo **sostituire `argmax` con la funzione `multinomial` di PyTorch**. Definiamo uan funzione `print_sampled_tokens` che **campiona 100 volte**, per  vedere le frequenze dei vari token:

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123) # seme per la riproducibilità
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")

print_sampled_tokens(probas)

Come possiamo vedere, la parola *forward* è stata campionata 544 volte su 1.000, ma anche altri token come *closer*, *inches* e *toward* sono stati selezionati occasionalmente.<br>
Ciò significa che, con la **sosituzione** che abbiamo fatto in `print_sampled_tokens`, il modello può **talvolta** generare frasi come:

"every effort moves you *toward*"

"every effort moves you *inches*"

"every effort moves you *closer*"

**invece** di "every effort moves you *forward*".

Possiamo <u>controllare ulteriormente la distribuzione</u> tramite un concetto chiamato **temperature scaling**, che consiste nel **dividere i logit per un valore maggiore di 0**:

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

Temperature <u>maggiori di 1</u> portano a distribuzioni di probabilità **più uniformi**; temperature <u>minori di 1</u> rendono la distribuzione più difforme e con picchi.<br>
Lo vediamo **confrontando** le probabilità originali con quelle scalate, per varie temperature diverse:

In [ ]:
# i valori delle temperature
temperatures = [1, 0.1, 5]  # originale, alta confidenza, bassa confidenza

# le probabilità scalate
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

⚠️ **Nota**: su Windows la cella seguente potrebbe chiudere il kernel — è il conflitto OpenMP descritto nella nota 📌 della sezione sul grafico delle loss. Con l'ambiente bonificato (o con la variabile `KMP_DUPLICATE_LIB_OK` della prima cella del notebook) la cella funziona regolarmente.

In [ ]:
import matplotlib.pyplot as plt
# plotting
x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f'Temperature = {T}')

ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.savefig("temperature-plot.pdf")
plt.show()

![](Figure_5_14_corretta.png)

In [ ]:
print_sampled_tokens(scaled_probas[1])

In [ ]:
print_sampled_tokens(scaled_probas[2])

Una temperatura pari a 1 rappresenta i punteggi originali non scalati.<br>
Abbassare la temperatura a 0.1 rende la distribuzione più netta: il token più probabile ("forward") ottiene una probabilità ancora più alta.<br>
Aumentare la temperatura a 5 rende la distribuzione più piatta e uniforme.<br>

Una temperatura pari a 1 equivale a non applicare alcuna scalatura.<br>
In questo caso, i token vengono selezionati con probabilità pari ai punteggi softmax originali tramite multinomial.
Per esempio, con temperatura 1, il token "forward" verrebbe scelto circa il 60% delle volte (come mostrato dal precedente plot).

Dallo stesso plot si vede anche che una temperatura molto bassa (come 0.1) produce distribuzioni estremamente sbilanciate: la funzione `multinomial` seleziona quasi sempre il token più probabile ("forward"), comportandosi quasi come `argmax`.
Al contrario, con temperatura 5 la distribuzione è molto più uniforme e altri token vengono selezionati più spesso.
Questo **aumenta la varietà del testo generato, ma può anche portare a risultati senza senso**.<br>
Per esempio, con temperatura 5, il testo generato può essere: "every effort moves you pizza" circa nel 4% dei casi.

**Esercizio**<br>
Usa la funzione `print_sampled_tokens` per campionare le frequenze delle probabilità softmax con le temperature mostrate nel plot precedente. Quanto spesso la parola "pizza" è campionata , con le varie temperature?

## Il campionamento *Top-k*

Il *temperature scaling* può portare a risultati grammaticalmente scorretti o completamente senza senso, come "every effort moves you pizza".

Il **top-k sampling**, <u>combinato</u> con *temperature scaling* e *probabilistic sampling*, può **migliorare i risultati**.
Con il *top-k sampling*, limitiamo i token selezionabili ai **k più probabili**, escludendo tutti gli altri (ponendo le loro probabilità a zero), come mostrato nella figura seguente:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/topk.webp" width=800px>

Se usiamo *top-k = 3* ci concentriamo sui tre token con i logit più alti e azzeriamo (impostando a $–inf$) quelli degli altri token <u>prima</u> di applicare la softmax.<br>
Ne risulta una distribuzione in cui **tutti i token non presenti nella top-k hanno probabilità zero**.<br>
[Nella figura i numeri sono troncati a due cifre decimali. I valori nelle righe softmax devono sommare a 1.]

In pratica, il *top-k sampling* **sostituisce i logit non selezionati con –inf**.<br>
Così, una volta applicata la softmax, la loro probabilità sarà 0, e le rimanenti probabilità si normalizzeranno a sommare 1.<br>
[E' un trucco che abbiamo già usato nel modulo di attenzione causale del notebook 3]

Il seguente codice implementa la procedura *top-k* mostrata nella figura, partendo dalla selezione dei token con i logit più alti:

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

# i valori di logit e gli ID dei token top-3 (in ordine decrescente):
print("logit top:", top_logits)
print("posizioni top:", top_pos)

Poi applichiamo `torch.where` di PyTorch per impostare a *–inf* i <u>logit inferiori al più basso valore nella top-3</u>.<br>
Il risultato sarà un vettore logit per il prossimo token (su un vocabolario di **nove token**).

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],  # identifica i logits inferiori al minimo del top-3
    input=torch.tensor(float("-inf")),             # assegna -inf a questi logits inferiori
    other=next_token_logits                        # per tutti gli altri token mantine i logits originali
)

print(new_logits)

> NOTA:
>
> **Un'implementazione alternativa**, leggermente più efficiente, della cella di codice precedente è la seguente:
>
> ```python
> new_logits = torch.full_like(                    # crea un tensore con i valori -inf
>    next_token_logits, -torch.inf
>)   
> new_logits[top_pos] = next_token_logits[top_pos] # copiai valori top k nel tensore -inf
> ```
> <br>
> Per maggiori dettagli, consultare https://github.com/rasbt/LLMs-from-scratch/discussions/326


Infine, applichiamo softmax per ottenere le probabilità dei prossimi token:

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

Come si vede, il risultato del top-3 è una **distribuzione con tre probabilità non nulle**.

Possiamo ora **combinare** il *temperature* scaling con la funzione `multinomial` per effettuare il campionamento (*sampling*) **solo tra questi tre token** e <u>generare così il token successivo</u>.<br>
Facciamo ciò modificando la funzione di generazione.

## Modificare la funzione di generazione testo

Come detto, combiniamo le tecniche di *temperature sampling* e *top-k sampling* per **modificare la funzione `generate_text_simple`** usata precedentemente e crearne una **nuova**. Le modifiche sono contrassegnate da **NEW**:

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # lo stesso ciclo for-loop di prima: calcola i logits e si focalizza solo sullo step dell'ultima volta
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # NEW: filtra i logits con il top_k sampling
        if top_k is not None:
            # mantiene solo i valori top_k
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # NEW: applica il temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # applica softmax per ottenere le probabilità
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # campiona dalla distribuzione
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        else:  # temperature scaling disabilitato
            # COME PRIMA: fa la selezione del prossimo token in modo "greedy",
            # cioè ottiene gli IDs delle entry del vocabolario con i logits più alti
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # interrompe in anticipo la generazione se incontra il token 'end-of-sequence'
            break

        # COME PRIMA: appende l'indice campionato alla sequenza corrente
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

Vediamo la funzione in azione:

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("testo generato:\n", token_ids_to_text(token_ids, tokenizer))

Il testo generato è **molto diverso da quello ottenuto in precedenza** con la vecchia funzione `generate_simple`, che era un passaggio memorizzato dal set di training.

> 📌 **NOTA D'AULA — Le manopole, dal vivo**<br>
> Il modo più efficace di spiegare temperatura e top-k non è descriverli: è **rilanciare la cella qui sopra cambiando solo i valori** — `temperature=0.1` (quasi deterministico), poi `1.5` (brioso, fino allo sgangherato); `top_k=5` contro `50` — si vede il testo cambiare carattere a ogni esecuzione.<br>
> Pochi secondi a giro su GPU: l'esperimento interattivo più economico del corso. L'Esercizio qui sotto chiede ai corsisti proprio questo.

**Esercizio 5.2**<br>
Sperimenta con temperature e valori di k diversi.<br>
Sulla base delle tue osservazioni, riesci a pensare a casi in cui siano preferibili temperature basse e valori di top-k ridotti?<br>
E casi in cui siano più adatte temperature alte e top-k elevati?<br>
(Suggerimento: riprendi questo esercizio anche alla fine del capitolo dopo aver caricato i pesi pre-addestrati da OpenAI.)

**Esercizio 5.3**<br>
Quali sono le combinazioni di impostazioni della funzione generate che forzano un comportamento deterministico, cioè disabilitano il sampling casuale e fanno sì che il modello produca sempre lo stesso output, proprio come fa la funzione generate_simple?

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>E in gpt-oss? — Temperatura e top-k: manopole identiche (più una nuova)</b><br><br>
La funzione <code>generate</code> appena scritta — temperatura, top-k, campionamento con <code>torch.multinomial</code> — usa <u>esattamente</u> le manopole con cui si pilota anche gpt-oss; nella pratica si aggiunge spesso il <b>top-p</b> (campionamento "a nucleo": invece dei k token migliori, si tengono i più probabili fino a coprire una probabilità cumulata p).<br><br>
La vera novità non è nel decoding ma <u>a monte</u>: il <b>reasoning effort</b> (low / medium / high), un'istruzione del formato Harmony che regola quanto ragionamento il modello produce nel canale <i>analysis</i> prima della risposta finale. Non tocca softmax né campionamento: cambia <u>che cosa</u> il modello sceglie di scrivere, non <u>come</u> viene estratto il token. Lo proveremo nel modulo dedicato.
</div>

# Caricare e salvare i pesi del modello in PyTorch

Nei capitoli precedenti abbiamo visto come **valutare numericamente l’avanzamento del training** e come **pre-addestrare un LLM da zero**.

Anche se sia il modello LLM che il dataset utilizzati erano **relativamente piccoli**, abbiamo visto come il pretraining degli LLM **richieda comunque un notevole sforzo computazionale**.

Per questo è fondamentale poter **salvare il modello**, <u>i suoi pesi</u>, in modo da <u>non dover rieseguire il training ogni volta che vogliamo riutilizzarlo in una nuova sessione</u>.

> Vediamo quindi come **salvare e caricare un modello pre-addestrato**.

Nel prossimo capitolo, caricheremo all’interno della nostra istanza `GPTModel` un modello GPT di OpenAI pre-addestrato e **più performante** .

La seguente figura fa il punto della situazione del nostro corso:

![](Figure_5_16_corretta.png)

Fortunatamente, **salvare un modello in PyTorch è piuttosto semplice**.<br>
Il <u>modo consigliato</u> è quello di **salvare lo `state_dict` del modello**, che è un dizionario che **associa ogni layer ai relativi parametri**, utilizzando la funzione `torch.save`:

In [ ]:
torch.save(model.state_dict(), "model.pth")

Abbiamo scelto come *filename* "model.pth". L’estensione **.pth** è una **convenzione** che si usa comunemente in PyTorch, anche se tecnicamente si potrebbe usare qualsiasi estensione.

Dopo aver salvato i pesi del modello tramite lo `state_dict`, possiamo **caricarli in una nuova istanza** del modello `GPTModel` con il seguente codice:

In [ ]:
model = GPTModel(GPT_CONFIG_124M)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval() # switch del modello a modalità evaluation per l'INFERENZA

Come visto nel notebook 4, il dropout aiuta a prevenire l’overfitting durante l’addestramento, escludendo casualmente alcuni neuroni nei layer.<br>
Tuttavia, durante l’inferenza non vogliamo perdere informazioni, quindi dobbiamo disattivare il dropout.<br>
Con `model.eval()` attiviamo la modalità di valutazione del modello, **disabilitando i layer di dropout**.

Se prevediamo di **continuare il pretraining** del modello in seguito — ad esempio usando la funzione `train_model_simple` definita in questo notebook - è allora **consigliabile salvare <u>anche</u> lo stato dell’ottimizzatore**.

Gli ottimizzatori adattivi come *AdamW* memorizzano <u>ulteriori parametri aggiuntivi</u> per ogni peso del modello.<br>
*AdamW* utilizza dati storici per regolare dinamicamente i learning rate di ciascun parametro.<br>
Se non salviamo anche lo stato dell’ottimizzatore, al momento del ripristino il **modello potrebbe imparare in modo subottimale, o addirittura non convergere**, perdendo la capacità di generare testo coerente.

In `torch.save`, possiamo salvare **sia lo stato del modello sia quello dell’ottimizzatore**:

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),            # salva i pesi
    "optimizer_state_dict": optimizer.state_dict(),    # salva i parametri aggiuntivi di AdamW
    },
    "model_and_optimizer.pth"
)

Possiamo poi **ripristinare** modello e ottimizzatore caricando i dati salvati con `torch.load` e applicando `load_state_dict`:

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

**Esercizio**<br>
Dopo aver salvato i pesi, carica il modello e l’ottimizzatore in un nuovo notebook Jupyter, e continua il pretraining per un'altra epoca usando la funzione `train_model_simple`.

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>E in gpt-oss? — Da <code>state_dict</code> a safetensors</b><br><br>
Il concetto appena visto — serializzare un dizionario di tensori — è lo stesso con cui viaggiano i pesi di gpt-oss; cambia il contenitore: non un <code>.pth</code> prodotto da <code>torch.save</code>, ma file <b>safetensors</b> (il formato standard di Hugging Face per i tensori: solo dati, niente codice eseguibile — più sicuro del <i>pickle</i> usato da <code>torch.save</code>), già quantizzati in MXFP4.<br><br>
Sotto il cofano, caricare gpt-oss significa comunque riempire uno <code>state_dict</code>: esattamente l'operazione che padroneggiamo qui. E naturalmente si distribuiscono i <u>pesi</u>, non lo stato di AdamW: quello serve solo a chi continua l'allenamento.
</div>

# Caricare i pesi pre-allenati di OpenAI

In precedenza abbiamo addestrato **un piccolo modello GPT-2 usando un dataset limitato**, costituito dal breve racconto “The Verdict”. Questo approccio ci ha permesso di concentrarci sui fondamenti, senza richiedere troppo tempo o potenza di calcolo.

Fortunatamente, OpenAI ha reso **pubblici i pesi dei modelli GPT-2**, eliminando così la necessità di spendere decine o centinaia di migliaia di dollari per riaddestrare il modello **su un ampio corpus documentale**.

Vediamo quindi come:
> **caricare questi pesi** all’interno della nostra classe `GPTModel` e usarli per generare testo.

Per “pesi” si intendono i **parametri** contenuti negli attributi `weight` dei layer *Linear* e *Embedding* di PyTorch.<br>
Li abbiamo **già visti** quando, durante il training, abbiamo usato **`model.parameters()`**.

Nel notebook 6 riutilizzeremo questi pesi pre-addestrati per eseguire **il fine-tuning** del modello su un compito di **classificazione testuale** e, nel notebook 7, per fargli **seguire istruzioni**, in stile ChatGPT.

OpenAI ha originariamente salvato i pesi di GPT-2 **usando [TensorFlow](https://www.tensorflow.org/)**, che dobbiamo installare per poterli caricare in Python.
Il codice seguente utilizza anche `tqdm`, una libreria per visualizzare una **barra di avanzamento** <u>durante il download</u>.

Puoi installare le librerie necessarie eseguendo questo comando **nel terminale**:

```cmd
python -m pip install tensorflow tqdm
```

In [ ]:
print("TensorFlow version:", version("tensorflow"))
print("tqdm version:", version("tqdm"))

**Nota**: in questo capitolo alcuni utenti potrebbero incontrare problemi legati alla **compatibilità di *TensorFlow***, in particolare su certi sistemi Windows.<br>
> TensorFlow ci serve qui **soltanto** per leggere i file dei pesi originali di OpenAI per GPT-2, che poi convertiamo in PyTorch.<br>
> In caso di problemi legati a TensorFlow, possiamo usare il codice alternativo qui sotto al posto del resto di questa sezione: è basato su pesi già convertiti in formato PyTorch, creati con lo stesso procedimento di conversione descritto di questo capitolo. Per i dettagli, si veda il notebook [../02_alternative_weight_loading/weight-loading-pytorch.ipynb](../02_alternative_weight_loading/weight-loading-pytorch.ipynb).


Il codice di download è piuttosto lungo e in gran parte *boilerplate*, quindi non lo riportiamo qui.<br>
Invece, scarichiamo direttamente il modulo Python `gpt_download.py` dal repository online di questo notebook.

In [ ]:
import os, urllib.request

file_name = "gpt2-small-124M.pth"
# file_name = "gpt2-medium-355M.pth"
# file_name = "gpt2-large-774M.pth"
# file_name = "gpt2-xl-1558M.pth"

# Configurazione del modello 124M, coerente con i pesi convertiti
# (contesto 1024 e qkv_bias=True, come richiesto dai pesi originali di OpenAI)
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True
}

url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)
    print(f"Downloaded to {file_name}")

gpt = GPTModel(BASE_CONFIG)
gpt.load_state_dict(torch.load(file_name, weights_only=True))
gpt.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device)

torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=BASE_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
import urllib.request
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch05/"
    "01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

La cella precedente **scarica un file dal repository GitHub di Raschka e lo salva nella cartella del notebook** — niente di più, niente di meno. 

Vediamo riga per riga:

- `import urllib.request`: carica il modulo della libreria standard per fare richieste HTTP (nessun pacchetto extra da installare).
- `url = (...)`: costruisce l'indirizzo del file. Le tre stringhe tra parentesi, una sotto l'altra senza virgole, vengono **concatenate automaticamente** da Python — è la *concatenazione implicita di letterali adiacenti*, un modo elegante per spezzare un URL lungo su più righe. Il risultato punta a `raw.githubusercontent.com`, il dominio che serve i file di GitHub "nudi e crudi" (il contenuto grezzo, senza l'interfaccia web attorno).
- `filename = url.split('/')[-1]`: spezza l'URL su ogni `/` e prende **l'ultimo pezzo**, cioè `gpt_download.py` — un idioma comune per ricavare il nome del file dal suo indirizzo.
- `urllib.request.urlretrieve(url, filename)`: fa il download vero e proprio e scrive il contenuto in un file locale chiamato `gpt_download.py`, nella cartella di lavoro del notebook.

L'output che vedi sotto — `('gpt_download.py', <http.client.HTTPMessage ...>)` — è semplicemente il **valore di ritorno** di `urlretrieve` mostrato da Jupyter (ultima espressione della cella): una tupla con il nome del file salvato e gli header HTTP della risposta. Non è un errore né un avviso: è la ricevuta della consegna.

**Perché il notebook lo fa**: come dice la cella markdown poco sopra, il codice che scarica e legge i checkpoint TensorFlow di OpenAI è lungo e di puro contorno (*boilerplate*, codice standard ripetitivo), quindi invece di incollarlo nel notebook lo si preleva come modulo già pronto — e infatti la cella successiva fa `from gpt_download import download_and_load_gpt2`. Due dettagli pratici: se il file esiste già viene **sovrascritto** senza chiedere (a differenza della cella dei `.pth`, che controllava con `os.path.exists`); e ora che è sul tuo disco puoi aprirlo in VS Code come un normale `.py` — vale la pena dargli un'occhiata, è il "dietro le quinte" di ciò che il capitolo usa subito dopo.

Dopo aver scaricato il file nella cartella locale della tua sessione Python, è buona norma verificarne rapidamente il contenuto per assicurarsi che sia stato salvato correttamente e contenga codice Python valido. In questo modo:

In [ ]:
# Verifica del file scaricato: dimensione plausibile, contenuto leggibile,
# e sintassi Python valida — SENZA eseguire il codice
import os, ast

print("Dimensione:", os.path.getsize("gpt_download.py"), "byte")
sorgente = open("gpt_download.py", encoding="utf-8").read()
ast.parse(sorgente)        # solleva SyntaxError se non è Python valido
print("Prime righe:\n", sorgente[:300])

A questo punto possiamo **importare la funzione `download_and_load_gpt2`** dal file *gpt_download.py*, che **caricherà in memoria le configurazioni architetturali (settings) e i pesi (params) del modello GPT-2**:

In [ ]:
# l'import da 'gpt_download.py' contenuto in questa directory
from gpt_download import download_and_load_gpt2
# alternativamente
# from llms_from_scratch.ch05 import download_and_load_gpt2

---

**Nota**

- In casi molto rari, la cella di codice sopra riportata potrebbe generare un errore `zsh: illegal hardware instruction python`, che potrebbe essere dovuto a un problema di installazione di TensorFlow sul computer.
- Un lettore ha scoperto che l'installazione di TensorFlow tramite `conda` ha risolto il problema in questo caso specifico, come menzionato [qui](https://github.com/rasbt/LLMs-from-scratch/discussions/273#discussioncomment-12367888)
- Puoi trovare ulteriori istruzioni in questo [tutorial di configurazione di Python](https://github.com/rasbt/LLMs-from-scratch/tree/main/setup/01_optional-python-setup-preferences#option-2-using-conda) supplementare.

---


Possiamo quindi scaricare i pesi del modello per il **modello a 124 milioni di parametri**.<br>
Eseguendo questo codice vengono scaricati i **sette file associati** al modello GPT-2 da 124 milioni di parametri. Occorrono circa 1-2 minuti su un PC Windows 4GHz / 16GBRAM:

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

📌 Nota
Se il codice di download non funziona, potrebbe dipendere da problemi di connessione, dal server o da cambiamenti nella modalità con cui OpenAI distribuisce i pesi open source.
In tal caso, visita il repository ufficiale del codice su:
https://github.com/rasbt/LLMs-from-scratch


<u>Assumendo</u> che il download sia andato a buon fine, ispezioniamo il **contenuto** di `settings` e `params`.<br>
Entrambi sono dizionari Python.<br>
* `settings` contiene i **parametri architetturali**, simili a quelli che abbiamo definito <u>manualmente</u> in `GPT_CONFIG_124M`
* `params` contiene **i tensori dei pesi effettivi**

In [ ]:
print("Settings:", settings)

In [ ]:
print("Parameter dictionary keys:", params.keys())

Abbiamo stampato solo le **chiavi** dei dizionari perchè stampare i contenuti dei pesi occuperebbe troppo spazio a video.<br>
Comunque, è possibile **ispezionare** questi tensori dei pesi in due modi:
* stampare <u>l'intero</u> dizionario con `print(params)` --> molto spazio a video, come detto
* **selezionare un singolo tensore** tramite la sua chiave (per esempio, per i pesi del layer di embedding --> **"wte"**)

In [ ]:
# prima opzione --> molto spazio a video
print(params)

In [ ]:
# seconda opzione:
print(params["wte"])
print("dimensioni del tensore dei 'token embedding weights':", params["wte"].shape)

I pesi che abbiamo caricato corrispondono al modello GPT-2 più piccolo, scaricato precedentemente con il setting:<br>
`download_and_load_gpt2(model_size="124M", models_dir="gpt2")`

**OpenAI ha reso disponibili anche i pesi per modelli più grandi: 355M, 774M e 1558M**.

L’<u>architettura di base</u> di tutti questi modelli è la STESSA (vedi la figura seguente), ECCETTO per:
* il numero di ripetizioni dei componenti (es. attention heads, transformer block)
* la dimensione degli embedding

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-sizes.webp?timestamp=123" width=700px>

Come si vede in figura, i modelli GPT-2 sono disponibili in versioni **da 124M a 1558M parametri**.
La struttura architetturale è identica, con variazioni nel numero di ripetizioni dei blocchi e nella dimensione degli embedding, come detto.

> Il codice successivo di questo notebook è **compatibile anche con questi modelli più grandi**.

Dopo aver caricato i pesi di GPT-2 in Python, dobbiamo **trasferirli dalla struttura `params` all’interno della nostra istanza `GPTModel`**.

Creiamo un dizionario `model_configs` che **elenca le differenze** tra i vari modelli GPT illustrati nella precedente figura.

In [ ]:
# definiamo le CONFIGURAZIONI dei modelli in un DIZIONARIO (per compattezza)
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

Supponiamo di voler caricare il modello più piccolo, "gpt2-small (124M)".<br>
Possiamo usare i relativi parametri di `model_configs` per aggiornare il nostro `GPT_CONFIG_124M`.

In [ ]:
# copiamo la configurazione base e modifichiamola con i setting di questo specifico modello
model_name = "gpt2-small (124M)"  # nome del modello di esempioe
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

Ricorderai che avevamo usato una <u>lunghezza di contesto pari a 256 token</u>, ma il GPT-2 originale di OpenAI è stato addestrato con **1.024 token**, quindi dobbiamo **aggiornare `NEW_CONFIG` di conseguenza**.

In [ ]:
NEW_CONFIG.update({"context_length": 1024})

Inoltre, OpenAI ha utilizzato <u>vettori di bias</u> nei layer lineari del modulo di attenzione multi-head per implementare i calcoli matriciali di query, chiavi e matrici di valori. **I vettori di bias non sono più comunemente utilizzati nei LLM** in quanto non migliorano le prestazioni di modellazione e sono quindi **superflui**. Tuttavia, poiché lavoriamo con <u>pesi pre-addestrati</u> (!), **dobbiamo adattare le impostazioni** per garantire la coerenza e abilitare questi vettori di bias:

In [ ]:
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

Possiamo ora usare il **dizionario `NEW_CONFIG` aggiornato** per inizializzare una **nuova istanza del modello**:

In [ ]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval();

Di default, l'istanza `GPTModel` è inizializzata con **pesi casuali** come pre-training.<br>
L’<u>ultimo passo</u> - per poter usare i *model weigths* di OpenAI - consiste nel **sovrascrivere questi pesi (casuali) con quelli caricati nel dizionario `params`**.

Per farlo, **definiamo** prima una piccola funzione `assign`, che:
* verifica **se due tensori o array (sinistri e destri) hanno la stessa dimensione (*shape*)**
* e **restituisce il tensore corretto** convertito in parametri PyTorch:

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

Poi, implementiamo la funzione `load_weights_into_gpt`, che **carica** i pesi dal dizionario `params` all’istanza gpt `GPTModel`.

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):                              # imposta i pesi del modello, sia POSIZIONALI che del
                                                                     # TOKEN EMBEDDING, a quelli specificati in 'params'

    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):                           # itera su ogni 'transformer block' del modello.

        q_w, k_w, v_w = np.split(                                    # la funzione 'n.split' è usata per dividere i pesi
                                                                     # attention e bias in 3 parti uguali per le componenti
                                                                     # query, key e value.
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])   # il modello GPT-2 originale di OpenAI RIUSAVA i pesi
                                                                       # dei token embedding nel layer di output per RIDURRE
                                                                       # il numero totale di parametri, un concetto noto
                                                                       # "weight tying"

Questa funzione `load_weights_into_gpt` esegue un'**attenta associazione** (*matching*) tra i pesi di OpenAI e quelli della nostra implementazione `GPTModel`

Per fare un <u>esempio specifico</u>, OpenAI ha memorizzato il tensore dei pesi del layer di proiezione di output per il primo blocco trasformatore in `params["blocks"][0]["attn"]["c_proj"]["w"])` Nella nostra implementazione questo tensore dei pesi corrisponde a `gpt.trf_blocks[b].att.out_proj.weight`, dove `gpt` è l'istanza `GPTModel`.

La scrittura di questa funzione ha richiesto vari tentativi, poiché OpenAI usa una convenzione di nomi leggermente diversa dalla nostra.<br>
Tuttavia, la funzione `assign` <u>ci avvisa</u> se tentiamo di associare tensori con dimensioni diverse.<br>
Inoltre, eventuali errori in questa funzione si noterebbero subito, perché il modello genererebbe **testo non coerente**.

Ora possiamo provare questa funzione `load_weights_into_gpt` e **caricare i pesi di OpenAI**:

In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device);

Se il modello è caricato <u>correttamente</u>, possiamo **usarlo subito** per generare **nuovo testo** con la funzione `generate` vista in precedenza.

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("testo di output:\n", token_ids_to_text(token_ids, tokenizer))

Possiamo **avere fiducia** che il caricamento sia avvenuto correttamente, poiché il modello produce **testo coerente e sensato**. Anche <u>un piccolo errore</u> in questo processo porterebbe a **output incoerenti o senza senso**.

Nei prossimi notebook, continueremo a lavorare con questo modello pre-addestrato, eseguendo il fine-tuning per classificare testi e seguirne le istruzioni.

Per un **metodo alternativo** per caricare i pesi dall'Hugging Face Hub, consulta [../02_alternative_weight_loading](../02_alternative_weight_loading).<br>
Se sei interessato a confrontare l'architettura GPT con l'architettura Llama (un popolare LLM sviluppato da Meta AI), consulta i contenuti bonus all'indirizzo [../07_gpt_to_llama](../07_gpt_to_llama).

In [ ]:
# ===== Confronto tra le due vie di caricamento =====
# PREREQUISITO: aver eseguito la cella che definisce load_weights_into_gpt
# (più sopra in questa sezione). Confronto MINIMO: un solo modello +
# un dizionario letto dal disco — qui NON si genera testo: si confrontano i PESI.

from gpt_download import download_and_load_gpt2
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

NEW_CONFIG = {
    "vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True
}

# Via TensorFlow: UN modello come contenitore della mappatura
gpt_tf = GPTModel(NEW_CONFIG)
load_weights_into_gpt(gpt_tf, params)

# Via .pth: nessun modello — il file È uno state_dict
sd_tf  = gpt_tf.state_dict()
sd_pth = torch.load("gpt2-small-124M.pth", map_location="cpu", weights_only=True)

print("Stessi nomi di parametri:", sd_tf.keys() == sd_pth.keys())
diversi = [k for k in sd_tf if not torch.equal(sd_tf[k].cpu(), sd_pth[k])]
print("Parametri diversi:", diversi if diversi else "nessuno — le due vie producono gli stessi pesi")

> `True` — "nessuno — le due vie producono gli stessi pesi": dimostrazione completa, in 5,5 secondi. Tutti i 149 tensori del modello — embedding, le dodici terne Q/K/V coi loro bias, i FeedForward, le norme, fino all'out_head — sono numericamente identici tra la conversione fatta al volo dai checkpoint TensorFlow e il .pth pre-convertito da Raschka. Da qui in poi è matematica, non fortuna: stessi pesi + stesso seed non possono che generare lo stesso testo. E nota le righe "File already exists and is up-to-date": il modulo ha solo riletto da disco, come previsto — zero ri-download.

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>E in gpt-oss? — Ieri i checkpoint TensorFlow, oggi Hugging Face</b><br><br>
Per GPT-2 (2019) siamo passati dai checkpoint TensorFlow di OpenAI, mappando i pesi uno a uno con <code>load_weights_into_gpt</code>. Per gpt-oss il punto di distribuzione è <b>Hugging Face</b> (il rimando qui sopra vale anche per lui), e la logica <i>from scratch</i> non cambia: l'implementazione PyTorch di riferimento pubblicata da OpenAI fa esattamente quel lavoro di mappatura — con una differenza che conosciamo dal notebook 4: <u>niente weight tying</u>, il layer di uscita ha i suoi pesi.<br><br>
In pratica, su hardware comune: il <b>20b</b> si carica con <code>transformers</code> in MXFP4 (~13 GB, entra in una GPU consumer) oppure dequantizzato in bf16 (~42 GB: servono offload o più GPU); il <b>120b</b> passa realisticamente da llama.cpp/Ollama con gli esperti in RAM. È la strada del modulo dedicato, dove rifaremo per gpt-oss ciò che <code>load_weights_into_gpt</code> fa qui per GPT-2.
</div>

**Esercizio 5.5**<br>
Calcolare le loss del training set e del validation set del `GPTModel` con i pesi pre-addestrati di OpenAI, sul dataset "The Verdict".

**Esercizio 5.6**<br>
Sperimentare con modelli GPT-2 di diverse dimensioni — ad esempio il più grande, da 1.558 milioni di parametri — e confrontare il testo generato con quello del modello da 124 milioni.



# Sintesi e messaggi takeaway
* Quando gli LLM generano testo, generano **un token alla volta**
* Per default, il token successivo è generato **convertendo** gli output del modello in **punteggi di probabilità** (*probability scores*) e **selezionando** dal vocabolario il token corrispondente al punteggio di probabilità **più alto**, un processo noto come "decodifica avida" (greedy decoding)
* Utilizzando il campionamento probabilistico (*probabilistic sampling*) e il *temperature scaling*, possiamo influenzare la **diversità e la coerenza del testo generato**
* Si possono utilizzare le perdite (*losses*) del training set e del validation set per **valutare la qualità** del testo generato dagli LLM <u>durante il training</u>
* Il pre-training di un LLM prevede la **modifica dei suoi pesi** per **minimizzare la perdita (*loss*) di training**
* Il ciclo di training per gli LLM è una **procedura standard** nel deep learning, che utilizza una ***cross entropy loss* convenzionale e un ottimizzatore *AdamW***
* Il pre-training di un LLM su <u>un corpus di testo di grandi dimensioni</u> **richiede molto tempo e risorse**, quindi possiamo **caricare pesi liberamente disponibili** come alternativa al pre-training del modello su un ampio dataset

- Vedi lo script [./gpt_train.py](./gpt_train.py), uno script autonomo per il training
- Lo script [./gpt_generate.py](./gpt_generate.py) carica i pesi pre-addestrati di OpenAI e genera testo a partire da un prompt
- Le soluzioni degli esercizi sono in [./exercise-solutions.ipynb](./exercise-solutions.ipynb)

<div style="background-color:#eaf3ff; border-left:6px solid #1565c0; padding:12px 14px; border-radius:4px">
🧩 <b>Riepilogo degli innesti — da GPT-2 a gpt-oss (capitolo 5)</b><br><br>
<table style="border-collapse:collapse">
<tr><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">Aspetto</th><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">GPT-2 124M (questo corso)</th><th style="border:1px solid #9bb8d3; padding:6px 10px; background:#d6e8fb; text-align:left">gpt-oss</th></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Funzione di perdita</td><td style="border:1px solid #9bb8d3; padding:6px 10px">cross-entropia sul prossimo token</td><td style="border:1px solid #9bb8d3; padding:6px 10px">identica (+ loss ausiliaria di bilanciamento per il router MoE)</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Dati di pre-training</td><td style="border:1px solid #9bb8d3; padding:6px 10px">"The Verdict", ~20.000 token</td><td style="border:1px solid #9bb8d3; padding:6px 10px">trilioni di token (inglese, STEM, codice)</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Epoche</td><td style="border:1px solid #9bb8d3; padding:6px 10px">10 sul nostro racconto</td><td style="border:1px solid #9bb8d3; padding:6px 10px">~1 passata sui dati</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Calcolo</td><td style="border:1px solid #9bb8d3; padding:6px 10px">minuti su una GPU</td><td style="border:1px solid #9bb8d3; padding:6px 10px">≈ 2,1 mln di ore-H100 (120b; ~10× meno il 20b)</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Contesto in training</td><td style="border:1px solid #9bb8d3; padding:6px 10px">ridotto a 256</td><td style="border:1px solid #9bb8d3; padding:6px 10px">esteso per fasi fino a 131.072 (YaRN)</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Decoding</td><td style="border:1px solid #9bb8d3; padding:6px 10px">greedy, temperatura, top-k</td><td style="border:1px solid #9bb8d3; padding:6px 10px">identico (+ top-p; reasoning effort via Harmony)</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Formato dei pesi</td><td style="border:1px solid #9bb8d3; padding:6px 10px"><code>state_dict</code> → .pth</td><td style="border:1px solid #9bb8d3; padding:6px 10px">safetensors in MXFP4</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Fonte dei pesi</td><td style="border:1px solid #9bb8d3; padding:6px 10px">checkpoint TensorFlow di OpenAI</td><td style="border:1px solid #9bb8d3; padding:6px 10px">Hugging Face</td></tr>
<tr><td style="border:1px solid #9bb8d3; padding:6px 10px">Allenabilità su PC</td><td style="border:1px solid #9bb8d3; padding:6px 10px">pre-training e fine-tuning pieni</td><td style="border:1px solid #9bb8d3; padding:6px 10px">QLoRA del 20b (~14 GB); 120b solo inferenza</td></tr>
</table>
<br>
Il confronto completo, con l'implementazione dei nuovi componenti, è sviluppato nel <b>modulo dedicato a gpt-oss</b> (notebook 8, <i>"Da GPT-2 a gpt-oss"</i>).
</div>

# Ottimizzazioni di Daily Dose of Data Science
**​A Memory-efficient technique to train large models** in DDoDS del 3 aprile 2026.<br>

Vedi anche - più in generale -[15 Ways to Optimize Neural Network Training (With Implementation)](https://www.dailydoseofds.com/15-ways-to-optimize-neural-network-training-with-implementation/).